# SELECT 데이터셋 검증 및 재선정

| 단계 | 할 일 | 남길 근거 |
|---|---|---|
| ① 경로 설정  | `select` 저장 구조·JSON 형식·매핑 확인 | 파일 수, 대표 JSON, 적용 규칙 |
| ② 구조적 오류 검사 | 필수 필드·연결 관계·bbox·이미지 파일 검사 | 오류 목록, 영향받은 이미지 수 |
| ③ 의심 사례 검증 | 중복 bbox·고중첩·제품 정보·구성 불일치 확인 | 원본 비교, 박스 시각화, 판정 이유 |
| ④ 편향 분석 | 클래스별 크기·위치·수량 분포 확인 | 분포와 해석 |
| ⑤ 재선정 | 확정 판정을 반영해 새 선정본 생성 | 제외 사유, 선정 전후 수량·클래스 분포 |

---
---
## 1. 경로 설정

### 1-1. 실행 환경과 경로설정

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

DATA_DIR = Path("/Users/codeit/Desktop/코드잇/project/pill-project/data")
SELECT_DIR = DATA_DIR / "select"
IMAGES_DIR = SELECT_DIR / "images"
ANNOTATIONS_DIR = SELECT_DIR / "annotations"
MAPPING_PATH = SELECT_DIR / "class_mapping.csv"

for folder in (SELECT_DIR, IMAGES_DIR, ANNOTATIONS_DIR):
    if not folder.is_dir():
        raise FileNotFoundError(f"폴더 경로를 확인하세요: {folder}")

def data_files(folder, suffixes):
    return sorted(
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in suffixes
        and "__MACOSX" not in p.relative_to(folder).parts
        and not p.name.startswith("._")
    )

json_paths = data_files(ANNOTATIONS_DIR, {".json"})
image_paths = data_files(IMAGES_DIR, {".png", ".jpg", ".jpeg"})
if not json_paths:
    raise ValueError("검사할 JSON 파일이 없습니다.")

print("검사 대상:", SELECT_DIR)
print(f"이미지 파일: {len(image_paths):,}개")
print(f"JSON 파일: {len(json_paths):,}개")
print("※ 파일 수 확인이며 정상 연결·중복·손상 검사는 아직 진행하지 않았습니다.")

inventory = pd.DataFrame([
    {"항목": name, "존재": (SELECT_DIR / name).is_file()}
    for name in ("manifest.csv", "saved_manifest.csv", "class_mapping.csv")
])
display(inventory)

class_mapping = (
    pd.read_csv(MAPPING_PATH, encoding="utf-8-sig", dtype=object)
    if MAPPING_PATH.is_file() else None
)
if class_mapping is not None:
    print(f"클래스 매핑: {len(class_mapping):,}행")
    print("매핑 컬럼:", class_mapping.columns.tolist())
    display(class_mapping.head(3))
else:
    print("클래스 매핑표 없음: 후속 매핑 검사는 검사 불가로 기록 예정")

### 1-2. 대표 JSON 구조 확인

In [ ]:
sample_path = json_paths[0]
with sample_path.open(encoding="utf-8-sig") as f:
    sample = json.load(f)

if not isinstance(sample, dict):
    raise TypeError("대표 JSON 최상위 구조가 dict가 아닙니다.")

print("표본 파일:", sample_path.relative_to(SELECT_DIR))
print("최상위 키:", list(sample))
for key in ("images", "annotations", "categories"):
    records = sample.get(key)
    print(f"\n[{key}] 자료형: {type(records).__name__}")
    if isinstance(records, list):
        print("항목 수:", len(records))
        if records:
            print(json.dumps(records[0], ensure_ascii=False, indent=2))
    else:
        print("예상한 list 형식이 아닙니다:", repr(records))

print("\n[source]")
print(json.dumps(sample.get("source"), ensure_ascii=False, indent=2))
print("\n1단계 구조 확인 실행 완료. 전체 오류 검사는 다음 단계에서 진행합니다.")

## 1단계 실행 결과 및 판단 기록

### 확인 결과
- 검사 대상: `/Users/codeit/Desktop/코드잇/project/pill-project/data/select`
- 이미지 파일: 10,240개
- JSON 파일: 10,240개
- 클래스 매핑표: 118행
- 위 수량은 파일 및 매핑표의 행 수이며,
  이미지 연결·중복·손상·매핑 정확성 검사는 아직 진행하지 않았다.
- 대표 JSON의 최상위 키는 `images`, `annotations`, `categories`, `source`이다.
- 대표 JSON에는 이미지 정보 1개, annotation 4개, category 4개가 포함되어 있다.
- 이미지 정보에는 ID, 파일명, 가로·세로 크기가 저장되어 있다.
- Annotation에는 bbox, 면적, 이미지 ID, 클래스 ID와 함께
  `drug_code`, `model_class_id`, `source_dl_idx`가 저장되어 있다.
- `source`에는 원본 Source 번호, 이미지 파일명, 내부 경로가 저장되어 있다.
- 클래스 매핑표를 정상적으로 불러왔으며 다음 컬럼을 확인했다.
  - code, category_id, model_class_id
  - aihub_dl_idx, codeit_train_class, id_difference

### 판단
- 대표 파일은 이미지 1장에 여러 알약의 annotation을 합친 구조이다.
- 전체 검사에서는 각 JSON의 모든 annotation을 검사해야 한다.
- 표본의 `K-000250`은 category_id=250, model_class_id=0,
  source_dl_idx=249로 저장되어 있다.
- 따라서 category_id와 source_dl_idx가 다르다는 이유만으로
  오류로 판단하지 않고 클래스 매핑표와 대조한다.
- 파일명에 포함된 알약 구성은 `source.original_image`를 기준으로 확인한다.
- 현재 결과는 대표 파일의 저장 구조를 확인한 것이며,
  전체 JSON·이미지의 정합성과 클래스 매핑 정확성은 아직 미판정이다.

### 다음 작업
- 전체 JSON을 읽어 검사에 사용할 공통 테이블을 만든다.
- JSON 읽기 실패, 필수 필드 누락, image_id 연결 오류부터 검사한다.

---
---
## 2. 구조적 오류 검사

### 2-1. 전체 JSON 읽기 및 기본 구조 검사

#### 검사 목적
- 전체 JSON을 읽어 이미지·annotation·category·출처 정보를 공통 테이블로 만든다.
- JSON 읽기 실패와 필수 필드 누락을 기록한다.
- 각 annotation의 image_id가 같은 JSON의 images.id에 연결되는지 확인한다.

#### 판정 기준
- JSON 읽기 실패, 예상 자료형 위반, 필수 필드 누락,
  존재하지 않는 image_id 참조는 구조적 오류로 기록한다.
- 필드 존재 여부와 실제 값의 유효성은 구분한다.
- bbox 좌표·면적·ID 중복·클래스 매핑·실제 PNG 검사는 후속 단계에서 진행한다.

In [ ]:
# 이후 검사에서도 재사용할 공통 테이블과 문제 목록
table_rows = {
    "images": [],
    "annotations": [],
    "categories": [],
    "source": [],
}
document_rows = []
issue_rows = []

# 현재 select 생성 형식에 필요한 필드
required_fields = {
    "images": {"id", "file_name", "width", "height"},
    "annotations": {
        "id", "image_id", "category_id", "bbox", "area",
        "drug_code", "model_class_id", "source_dl_idx",
    },
    "categories": {"id", "name"},
    "source": {"source_number", "original_image", "original_path"},
}


def add_structure_issue(json_file, item, detail, record_index=None):
    issue_rows.append({
        "json_file": json_file,
        "구분": "오류",
        "검사항목": item,
        "record_index": record_index,
        "상세": detail,
    })


for path in json_paths:
    json_file = path.relative_to(SELECT_DIR).as_posix()
    issue_start = len(issue_rows)
    read_ok = False

    try:
        with path.open(encoding="utf-8-sig") as f:
            data = json.load(f)
        read_ok = True
    except (OSError, ValueError) as e:
        add_structure_issue(
            json_file, "JSON 읽기 실패",
            f"{type(e).__name__}: {e}"
        )
        data = None

    # 파일 읽기 성공과 최상위 구조 정상 여부를 구분
    root_ok = read_ok and isinstance(data, dict)

    if read_ok and not root_ok:
        add_structure_issue(
            json_file, "최상위 자료형 오류",
            f"예상: dict / 실제: {type(data).__name__}"
        )

    if root_ok:
        parsed = {}

        for section, required in required_fields.items():
            if section not in data:
                add_structure_issue(
                    json_file, "필수 섹션 누락", section
                )
                parsed[section] = []
                continue

            value = data[section]
            expected_type = dict if section == "source" else list

            if not isinstance(value, expected_type):
                add_structure_issue(
                    json_file, "섹션 자료형 오류",
                    f"{section}: 예상 {expected_type.__name__}, "
                    f"실제 {type(value).__name__}"
                )
                parsed[section] = []
                continue

            records = [value] if section == "source" else value
            parsed[section] = []

            # 현재 select는 이미지 1장당 JSON 1개
            if section == "images" and len(records) != 1:
                add_structure_issue(
                    json_file, "이미지 정보 개수 오류",
                    f"예상: 1 / 실제: {len(records)}"
                )

            if section == "annotations" and not records:
                add_structure_issue(
                    json_file, "빈 annotation 목록",
                    "현재 선정본에서 annotation이 없는 JSON"
                )

            for index, record in enumerate(records):
                if not isinstance(record, dict):
                    add_structure_issue(
                        json_file, "기록 자료형 오류",
                        f"{section}[{index}]: "
                        f"{type(record).__name__}",
                        index,
                    )
                    continue

                missing = sorted(required - record.keys())
                if missing:
                    add_structure_issue(
                        json_file, "필수 필드 누락",
                        f"{section}[{index}]: {missing}",
                        index,
                    )

                # 원래 값을 유지하고 JSON 경로·기록 순번을 함께 저장
                table_rows[section].append({
                    **record,
                    "_json_file": json_file,
                    "_record_index": index,
                })
                parsed[section].append((index, record))

        # 같은 JSON 내부에서 annotation → image 연결 검사
        image_ids = [
            record["id"]
            for _, record in parsed["images"]
            if "id" in record
        ]

        for index, ann in parsed["annotations"]:
            # image_id 누락은 위의 필수 필드 검사에서 이미 기록
            if "image_id" not in ann:
                continue

            if ann["image_id"] not in image_ids:
                add_structure_issue(
                    json_file, "image_id 연결 오류",
                    f"annotations[{index}].image_id="
                    f"{ann['image_id']!r}, "
                    f"같은 JSON의 images.id={image_ids!r}",
                    index,
                )

    document_rows.append({
        "json_file": json_file,
        "읽기성공": read_ok,
        "최상위구조정상": root_ok,
        "문제건수": len(issue_rows) - issue_start,
    })


# 후속 단계에서 그대로 재사용
documents_df = pd.DataFrame(document_rows)
images_df = pd.DataFrame(table_rows["images"])
annotations_df = pd.DataFrame(table_rows["annotations"])
categories_df = pd.DataFrame(table_rows["categories"])
sources_df = pd.DataFrame(table_rows["source"])

structure_issues = pd.DataFrame(
    issue_rows,
    columns=[
        "json_file", "구분", "검사항목", "record_index", "상세"
    ],
)

print("========== JSON 기본 구조 검사 ==========")
print(f"검사 JSON: {len(documents_df):,}개")
print(f"읽기 성공: {documents_df['읽기성공'].sum():,}개")
print(f"읽기 실패: {(~documents_df['읽기성공']).sum():,}개")
print(f"문제 기록: {len(structure_issues):,}건")
print(
    f"문제가 기록된 JSON: "
    f"{structure_issues['json_file'].nunique():,}개"
)

print("\n========== 공통 테이블 ==========")
display(pd.DataFrame([
    {"테이블": "images_df", "행 수": len(images_df)},
    {"테이블": "annotations_df", "행 수": len(annotations_df)},
    {"테이블": "categories_df", "행 수": len(categories_df)},
    {"테이블": "sources_df", "행 수": len(sources_df)},
]))

if structure_issues.empty:
    print("\n이번에 검사한 구조·필드 존재·이미지 연결에서 오류 없음")
else:
    print("\n========== 검사 항목별 문제 수 ==========")
    display(
        structure_issues.groupby("검사항목")
        .agg(
            문제건수=("json_file", "size"),
            영향JSON수=("json_file", "nunique"),
        )
        .reset_index()
    )

    print("\n========== 문제 표본: 최대 20건 ==========")
    display(structure_issues.head(20))

print("\n※ 필드 값의 유효성·ID 중복·bbox·PNG 검사는 아직 미진행")

### 2-2. ID 중복 및 필드 값 유효성 검사

In [ ]:
import math
from numbers import Real

# 이번 단계의 문제 목록
id_issue_rows = []


def is_integer_number(value):
    """문자열·bool을 제외한 유한한 정수값인지 확인."""
    return (
        isinstance(value, Real)
        and not isinstance(value, bool)
        and math.isfinite(value)
        and float(value).is_integer()
    )


def add_id_issue(row, item, detail):
    id_issue_rows.append({
        "json_file": row["_json_file"],
        "record_index": row["_record_index"],
        "검사항목": item,
        "상세": detail,
    })


# 1. ID 값의 자료형·유한성·정수 여부
id_checks = [
    ("images", images_df, ["id"]),
    ("annotations", annotations_df, ["id", "image_id", "category_id"]),
    ("categories", categories_df, ["id"]),
]

for section, table, fields in id_checks:
    for field in fields:
        invalid = ~table[field].map(is_integer_number)

        for _, row in table.loc[invalid].iterrows():
            add_id_issue(
                row,
                "ID 값 형식 오류",
                f"{section}.{field}={row[field]!r}",
            )


# 2. JSON 이미지 크기: 양의 정수값이어야 함
for field in ["width", "height"]:
    valid = images_df[field].map(
        lambda value: is_integer_number(value) and value > 0
    )

    for _, row in images_df.loc[~valid].iterrows():
        add_id_issue(
            row,
            "JSON 이미지 크기 오류",
            f"{field}={row[field]!r}",
        )


# 3. Annotation ID 중복
# select는 저장할 때 전체 annotation에 고유 ID를 부여한 형식
valid_annotations = annotations_df.loc[
    annotations_df["id"].map(is_integer_number)
]

duplicate_annotations = valid_annotations.loc[
    valid_annotations["id"].duplicated(keep=False)
]

for _, row in duplicate_annotations.iterrows():
    add_id_issue(
        row,
        "Annotation ID 중복",
        f"annotation_id={row['id']!r}, "
        f"image_id={row['image_id']!r}, "
        f"category_id={row['category_id']!r}, "
        f"bbox={row['bbox']!r}",
    )


# 4. 이미지 ID 중복
# select는 이미지별 JSON 1개이며 이미지 ID도 전체에서 고유해야 함
valid_images = images_df.loc[
    images_df["id"].map(is_integer_number)
]

duplicate_images = valid_images.loc[
    valid_images["id"].duplicated(keep=False)
]

for _, row in duplicate_images.iterrows():
    add_id_issue(
        row,
        "이미지 ID 중복",
        f"image_id={row['id']!r}, "
        f"file_name={row['file_name']!r}",
    )


# 5. 같은 file_name에 서로 다른 이미지 ID가 연결되는지 검사
# 파일명 형식이 잘못된 기록은 별도로 남김
valid_filename = valid_images["file_name"].map(
    lambda value: isinstance(value, str) and bool(value.strip())
)

for _, row in valid_images.loc[~valid_filename].iterrows():
    add_id_issue(
        row,
        "이미지 파일명 형식 오류",
        f"file_name={row['file_name']!r}",
    )

filename_images = valid_images.loc[valid_filename]

filename_id_counts = (
    filename_images.groupby("file_name")["id"].nunique()
)
conflict_names = filename_id_counts[
    filename_id_counts > 1
].index

for _, row in filename_images.loc[
    filename_images["file_name"].isin(conflict_names)
].iterrows():
    add_id_issue(
        row,
        "동일 파일명의 이미지 ID 충돌",
        f"file_name={row['file_name']!r}, "
        f"image_id={row['id']!r}",
    )


# 6. 결과 출력 및 후속 단계용 문제 표 보관
id_issues = pd.DataFrame(
    id_issue_rows,
    columns=["json_file", "record_index", "검사항목", "상세"],
)

print("========== ID 및 기본 값 검사 ==========")
print(f"이미지 기록: {len(images_df):,}개")
print(f"Annotation 기록: {len(annotations_df):,}개")
print(f"문제 기록: {len(id_issues):,}건")
print(
    f"문제가 기록된 JSON: "
    f"{id_issues['json_file'].nunique():,}개"
)

if id_issues.empty:
    print("\n이번 검사에서 오류 없음")
    print("[체크 가능] Annotation ID 중복 검사")
    print("[체크 가능] 이미지 ID 중복 검사")
else:
    display(
        id_issues.groupby("검사항목")
        .agg(
            문제기록수=("json_file", "size"),
            영향JSON수=("json_file", "nunique"),
        )
        .reset_index()
    )
    display(id_issues.head(20))
    print("\n※ 중복 문제기록수는 해당 ID를 가진 기록 수입니다.")

print("\n※ category_id의 클래스 매핑 정확성은 아직 미검사")
print("※ width·height는 JSON 값만 검사했으며 실제 PNG와 비교 전")

### 2-3. bbox 면적·좌표·크기 검사

In [ ]:
import math
from numbers import Real

bbox_issue_rows = []
bbox_check_rows = []

image_lookup = images_df.set_index("id")


def is_finite_number(value):
    """bool을 제외한 유한한 숫자인지 확인."""
    return (
        isinstance(value, Real)
        and not isinstance(value, bool)
        and math.isfinite(value)
    )


def add_bbox_issue(row, item, detail):
    bbox_issue_rows.append({
        "json_file": row["_json_file"],
        "annotation_id": row["id"],
        "image_id": row["image_id"],
        "drug_code": row["drug_code"],
        "check_item": item,
        "detail": detail,
    })


for _, row in annotations_df.iterrows():
    bbox = row["bbox"]
    area = row["area"]

    check = {
        "json_file": row["_json_file"],
        "annotation_id": row["id"],
        "image_id": row["image_id"],
        "bbox_format_ok": False,
        "coordinates_ok": None,
        "boundary_ok": None,
        "area_match": None,
    }

    # 1. bbox의 자료형과 길이 검사
    bbox_format_ok = isinstance(bbox, list) and len(bbox) == 4

    if not bbox_format_ok:
        add_bbox_issue(
            row,
            "bbox 형식 오류",
            f"숫자 4개로 된 list가 필요함: {bbox!r}",
        )

    elif not all(is_finite_number(value) for value in bbox):
        add_bbox_issue(
            row,
            "bbox 숫자 오류",
            f"문자열·bool·NaN·무한대 등 확인: {bbox!r}",
        )
        bbox_format_ok = False

    check["bbox_format_ok"] = bbox_format_ok

    # 2. area 자체의 값 검사
    area_ok = is_finite_number(area) and area > 0

    if not area_ok:
        add_bbox_issue(
            row,
            "area 값 오류",
            f"양의 유한한 숫자가 필요함: {area!r}",
        )

    if bbox_format_ok:
        x, y, w, h = map(float, bbox)

        # 3. 음수 좌표와 0 이하 크기 검사
        coordinates_ok = (
            x >= 0 and y >= 0 and w > 0 and h > 0
        )
        check["coordinates_ok"] = coordinates_ok

        if not coordinates_ok:
            add_bbox_issue(
                row,
                "음수 좌표 또는 0 이하 크기",
                f"x={x}, y={y}, w={w}, h={h}",
            )

        # 4. JSON 이미지 크기 기준 범위 검사
        image = image_lookup.loc[row["image_id"]]
        width = float(image["width"])
        height = float(image["height"])

        right = x + w
        bottom = y + h

        # 좌표·크기가 잘못된 박스는 범위 정상으로 판정하지 않음
        if coordinates_ok:
            boundary_ok = (
                math.isfinite(right)
                and math.isfinite(bottom)
                and right <= width
                and bottom <= height
            )
            check["boundary_ok"] = boundary_ok

            if not boundary_ok:
                add_bbox_issue(
                    row,
                    "bbox 범위 초과",
                    f"오른쪽 끝={right}, 이미지 너비={width}; "
                    f"아래쪽 끝={bottom}, 이미지 높이={height}",
                )

        # 5. area와 w×h 비교
        expected_area = w * h

        if not math.isfinite(expected_area):
            add_bbox_issue(
                row,
                "bbox 면적 계산 오류",
                f"w×h가 유한하지 않음: w={w}, h={h}",
            )

        elif area_ok:
            area_match = math.isclose(
                float(area),
                expected_area,
                rel_tol=1e-9,
                abs_tol=1e-6,
            )
            check["area_match"] = area_match

            if not area_match:
                add_bbox_issue(
                    row,
                    "bbox 면적 불일치",
                    f"area={area}, w×h={expected_area}, "
                    f"차이={float(area) - expected_area}",
                )

    bbox_check_rows.append(check)


# 후속 단계에서도 재사용할 검사 결과
bbox_checks = pd.DataFrame(bbox_check_rows)

bbox_issues = pd.DataFrame(
    bbox_issue_rows,
    columns=[
        "json_file",
        "annotation_id",
        "image_id",
        "drug_code",
        "check_item",
        "detail",
    ],
)

# 출력용 항목 이름
check_labels = {
    "bbox_format_ok": "bbox 형식",
    "coordinates_ok": "음수 좌표·0 이하 크기",
    "boundary_ok": "이미지 범위",
    "area_match": "면적 일치",
}

bbox_summary = pd.DataFrame([
    {
        "검사 항목": label,
        "검사 완료": int(bbox_checks[column].notna().sum()),
        "검사 불가": int(bbox_checks[column].isna().sum()),
        "정상": int(bbox_checks[column].eq(True).sum()),
        "이상": int(bbox_checks[column].eq(False).sum()),
    }
    for column, label in check_labels.items()
])

print("========== bbox 검사 ==========")
print(f"전체 Annotation: {len(annotations_df):,}개")
print(f"문제 기록: {len(bbox_issues):,}건")
print(
    f"문제가 있는 Annotation: "
    f"{bbox_issues['annotation_id'].nunique():,}개"
)
print(
    f"영향받은 이미지: "
    f"{bbox_issues['image_id'].nunique():,}장"
)

print("\n========== 항목별 검사 결과 ==========")
display(bbox_summary)

if bbox_issues.empty and bbox_summary["검사 불가"].sum() == 0:
    print("\n이번 검사에서 오류 없음")
    print("[체크 가능] bbox 면적 불일치 검사")
    print("[체크 가능] bbox 범위 초과 검사")
    print("[체크 가능] 음수 좌표·0 이하 크기 검사")

else:
    if not bbox_issues.empty:
        print("\n========== 문제 항목별 현황 ==========")
        display(
            bbox_issues.groupby("check_item")
            .agg(
                문제기록수=("annotation_id", "size"),
                영향Annotation수=("annotation_id", "nunique"),
                영향이미지수=("image_id", "nunique"),
            )
            .reset_index()
        )

        print("\n========== 문제 표본: 최대 20건 ==========")
        display(bbox_issues.head(20))

    print("\n오류 또는 검사 불가 항목을 확인해야 합니다.")

print("\n※ 범위 검사는 JSON의 width·height 기준입니다.")
print("※ 실제 PNG 크기 및 실제 알약 위치와의 비교는 아직 미진행입니다.")
print("※ area는 선정본 생성 시 재계산한 값으로, 원본 area 검증과 구분합니다.")

### 2-4. 같은 이미 안의 동일 bbox·과도한 중첩 검사

| 점검표 항목 | 이번 검사 기준 | 결과 해석 |
|---|---|---|
| 동일 bbox 반복 | 같은 이미지에서 `[x, y, w, h]`가 완전히 일치 | 중복 라벨 또는 클래스 충돌 후보 |
| 과도한 중첩 | 두 박스의 IoU가 0.8 이상 | 시각 확인이 필요한 후보 |
| 작은 박스 중첩 비율 | 작은 박스 면적의 90% 이상이 다른 박스와 겹침 | 검토 후보 |

In [ ]:
from itertools import combinations

IOU_THRESHOLD = 0.8
SMALL_BOX_OVERLAP_THRESHOLD = 0.9

overlap_rows = []
checked_pair_count = 0


def calculate_bbox_overlap(box_a, box_b):
    """유효한 [x, y, w, h] 박스의 두 가지 중첩 비율 계산."""
    ax, ay, aw, ah = map(float, box_a)
    bx, by, bw, bh = map(float, box_b)

    intersection_width = max(
        0.0, min(ax + aw, bx + bw) - max(ax, bx)
    )
    intersection_height = max(
        0.0, min(ay + ah, by + bh) - max(ay, by)
    )

    intersection_area = intersection_width * intersection_height

    area_a = aw * ah
    area_b = bw * bh
    union_area = area_a + area_b - intersection_area
    smaller_area = min(area_a, area_b)

    return {
        "iou": intersection_area / union_area,
        "small_box_overlap": intersection_area / smaller_area,
    }


# 앞 단계의 bbox 검사에서 오류·검사 불가가 없어야 진행
bbox_check_columns = [
    "bbox_format_ok",
    "coordinates_ok",
    "boundary_ok",
    "area_match",
]

if (
    bbox_checks.empty
    or not bbox_checks[bbox_check_columns].eq(True).all().all()
):
    raise ValueError(
        "앞 단계의 bbox 오류 또는 검사 불가 항목을 먼저 확인해 주세요."
    )


# 같은 이미지 안의 annotation끼리 비교
for image_id, group in annotations_df.groupby("image_id", sort=False):
    records = group.to_dict("records")

    for ann_a, ann_b in combinations(records, 2):
        checked_pair_count += 1

        same_bbox = ann_a["bbox"] == ann_b["bbox"]
        same_class = ann_a["category_id"] == ann_b["category_id"]

        metrics = calculate_bbox_overlap(
            ann_a["bbox"], ann_b["bbox"]
        )

        high_iou = metrics["iou"] >= IOU_THRESHOLD
        high_small_box_overlap = (
            metrics["small_box_overlap"]
            >= SMALL_BOX_OVERLAP_THRESHOLD
        )

        # 세 조건 중 하나라도 충족하면 후보로 기록
        if not (same_bbox or high_iou or high_small_box_overlap):
            continue

        reasons = []

        if same_bbox:
            reasons.append(
                "동일 bbox·동일 클래스"
                if same_class
                else "동일 bbox·다른 클래스"
            )

        if high_iou:
            reasons.append("높은 IoU")

        if high_small_box_overlap:
            reasons.append("작은 박스 중첩 비율 높음")

        overlap_rows.append({
            "json_file": ann_a["_json_file"],
            "image_id": image_id,
            "annotation_id_a": ann_a["id"],
            "annotation_id_b": ann_b["id"],
            "drug_code_a": ann_a["drug_code"],
            "drug_code_b": ann_b["drug_code"],
            "bbox_a": ann_a["bbox"],
            "bbox_b": ann_b["bbox"],
            "same_bbox": same_bbox,
            "same_class": same_class,
            "iou": metrics["iou"],
            "small_box_overlap": metrics["small_box_overlap"],
            "high_iou": high_iou,
            "high_small_box_overlap": high_small_box_overlap,
            "check_reason": " / ".join(reasons),
        })


# 한 박스 쌍은 조건이 여러 개에 해당해도 한 행으로 저장
overlap_candidates = pd.DataFrame(
    overlap_rows,
    columns=[
        "json_file",
        "image_id",
        "annotation_id_a",
        "annotation_id_b",
        "drug_code_a",
        "drug_code_b",
        "bbox_a",
        "bbox_b",
        "same_bbox",
        "same_class",
        "iou",
        "small_box_overlap",
        "high_iou",
        "high_small_box_overlap",
        "check_reason",
    ],
)

criteria = {
    "동일 bbox": "same_bbox",
    f"IoU {IOU_THRESHOLD:.0%} 이상": "high_iou",
    (
        f"작은 박스 중첩 "
        f"{SMALL_BOX_OVERLAP_THRESHOLD:.0%} 이상"
    ): "high_small_box_overlap",
}

summary_rows = []

for label, column in criteria.items():
    selected = overlap_candidates.loc[
        overlap_candidates[column].eq(True)
    ]

    summary_rows.append({
        "검사 항목": label,
        "후보쌍수": len(selected),
        "영향이미지수": selected["image_id"].nunique(),
    })

overlap_summary = pd.DataFrame(summary_rows)

print("========== 동일 bbox·중첩 검사 ==========")
print(f"검사 이미지: {annotations_df['image_id'].nunique():,}장")
print(f"비교한 박스 쌍: {checked_pair_count:,}쌍")
print(f"전체 후보: {len(overlap_candidates):,}쌍")
print(
    f"후보가 있는 이미지: "
    f"{overlap_candidates['image_id'].nunique():,}장"
)

display(overlap_summary)

if overlap_candidates.empty:
    print("\n이번 기준에 해당하는 후보 없음")
    print("[체크 가능] 동일 bbox 반복 — 없음")
    print("[체크 가능] 과도한 중첩 — 설정한 두 기준에서 후보 없음")

else:
    print("\n========== 검토 후보: 최대 20쌍 ==========")

    preview_columns = [
        "json_file",
        "annotation_id_a",
        "annotation_id_b",
        "drug_code_a",
        "drug_code_b",
        "bbox_a",
        "bbox_b",
        "iou",
        "small_box_overlap",
        "check_reason",
    ]

    display(
        overlap_candidates.sort_values(
            ["same_bbox", "small_box_overlap", "iou"],
            ascending=False,
        )[preview_columns].head(20)
    )

    if not overlap_candidates["same_bbox"].eq(True).any():
        print("\n[체크 가능] 동일 bbox 반복 — 없음")

    print("\n후보가 발견된 항목은 시각 확인 후 최종 판정합니다.")

print("\n※ 항목별 후보 수에는 같은 박스 쌍이 중복 집계될 수 있습니다.")
print("※ 전체 후보 수는 중복 집계하지 않은 박스 쌍 수입니다.")
print("※ 원본 데이터는 수정하거나 삭제하지 않았습니다.")

중복 집계를 제외한 검토 대상 현황

| 검토 대상 | 이미지 수 | 우선순위 |
|---|---:|---|
| bbox가 완전히 같음 | 67장 | 1 |
| 완전히 같지는 않지만 IoU 0.8 이상 | 3장 | 2 |
| 위 두 경우 외에 작은 박스 중첩 비율 90% 이상 | 24장 | 3 |
| **합계** | **94장** | |

### 동일 bbox가 발견된 67장 검토 
- 회색 박스: 해당 이미지의 전체 annotation
- 왼쪽 빨간 박스: 후보 A
- 오른쪽 파란 박스: 후보 B
- 아래 코드는 한 번에 3장씩 표시

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# 동일 bbox 후보만 검토 대상으로 구성
same_bbox_review = (
    overlap_candidates.loc[
        overlap_candidates["same_bbox"].eq(True)
    ]
    .reset_index(drop=True)
    .copy()
)
same_bbox_review.insert(
    0, "review_number", range(1, len(same_bbox_review) + 1)
)

# select 내부 실제 이미지 경로 조회표
review_image_paths = {}
for path in SELECT_DIR.rglob("*"):
    if path.is_file() and path.suffix.lower() in {
        ".png", ".jpg", ".jpeg", ".webp"
    }:
        review_image_paths.setdefault(path.name, []).append(path)

review_image_lookup = images_df.set_index("id")
review_annotation_groups = {
    image_id: group
    for image_id, group in annotations_df.groupby("image_id")
}


def resolve_review_image(file_name):
    """JSON에 저장된 경로를 우선 사용하고, 없으면 파일명으로 조회."""
    relative_path = Path(file_name)

    for candidate in [
        SELECT_DIR / relative_path,
        SELECT_DIR / "images" / relative_path,
    ]:
        if candidate.is_file():
            return candidate

    matches = review_image_paths.get(relative_path.name, [])

    if len(matches) != 1:
        raise ValueError(
            f"이미지 경로를 확정할 수 없음: {file_name} "
            f"(파일명 일치 {len(matches)}개)"
        )

    return matches[0]


def show_same_bbox_page(page=1, page_size=3, review_table=None):
    if review_table is None:
        review_table = same_bbox_review

    total = len(review_table)
    total_pages = (total + page_size - 1) // page_size

    if page_size < 1 or not 1 <= page <= total_pages:
        print(f"page는 1~{total_pages}, page_size는 1 이상으로 지정하세요.")
        return

    start = (page - 1) * page_size
    batch = review_table.iloc[start:start + page_size]

    print(f"동일 bbox 검토: {page}/{total_pages}페이지")
    display(batch[[
        "review_number", "json_file",
        "annotation_id_a", "drug_code_a",
        "annotation_id_b", "drug_code_b",
        "same_class",
    ]])

    fig, axes = plt.subplots(
        len(batch), 2,
        figsize=(12, 7 * len(batch)),
        squeeze=False,
    )

    for row_index, (_, candidate) in enumerate(batch.iterrows()):
        image_id = candidate["image_id"]
        image_info = review_image_lookup.loc[image_id]

        try:
            image_path = resolve_review_image(image_info["file_name"])
            with Image.open(image_path) as image:
                image_rgb = image.convert("RGB")
        except Exception as error:
            for ax in axes[row_index]:
                ax.axis("off")
            print(
                f"검토 {candidate['review_number']}번 표시 실패: {error}"
            )
            continue

        all_annotations = review_annotation_groups[image_id]

        for column_index, (side, color) in enumerate([
            ("a", "red"), ("b", "blue")
        ]):
            ax = axes[row_index, column_index]
            ax.imshow(image_rgb)

            # 전체 박스와 annotation ID 표시
            for _, annotation in all_annotations.iterrows():
                x, y, w, h = annotation["bbox"]
                ax.add_patch(Rectangle(
                    (x, y), w, h,
                    fill=False,
                    edgecolor="gray",
                    linewidth=1,
                ))
                ax.text(
                    x, y + h,
                    f"ID {annotation['id']}",
                    fontsize=8,
                    color="black",
                    va="bottom",
                    bbox=dict(
                        facecolor="white", alpha=0.7, edgecolor="none"
                    ),
                )

            # 문제의 두 박스를 화면별로 하나씩 강조
            x, y, w, h = candidate[f"bbox_{side}"]
            ax.add_patch(Rectangle(
                (x, y), w, h,
                fill=False,
                edgecolor=color,
                linewidth=2.5,
            ))

            ax.set_title(
                f"Review {candidate['review_number']} | {side.upper()}\n"
                f"ID {candidate[f'annotation_id_{side}']} | "
                f"{candidate[f'drug_code_{side}']}",
                color=color,
                fontsize=11,
            )
            ax.axis("off")

    plt.tight_layout()
    plt.show()


# 첫 3장 표시
show_same_bbox_page(page=1)


In [ ]:
page_size = 3
total_pages = (len(same_bbox_review) + page_size - 1) // page_size

for page in range(2, total_pages + 1):
    show_same_bbox_page(page=page, page_size=page_size)

#### 동일 bbox 시각 검토 결과

| 대상 | 검토 결과 | 처리 상태 |
|---|---|---|
| 동일 bbox 67장 | 한 알약에 두 박스가 겹친 것을 전수 확인 | 수정 대기 |
| 나머지 중첩 27장 | 아직 미확인 | 시각 검토 예정 |

- 동일 bbox 후보 67장을 모두 시각 검토했다.
- 67장 모두 실제 알약 하나의 위치에 두 annotation의 박스가
  완전히 겹쳐 있는 것을 확인했다.
- 위치 중복 오류로 기록하며, 수정은 아직 수행하지 않았다.
- 어느 annotation의 위치를 수정해야 하는지는 원본 라벨 및
  클래스 정보를 대조한 뒤 결정한다.
- 다음 작업: 동일 bbox를 제외한 나머지 중첩 후보 27장 시각 검토.

### 동일 bbox를 제외한 중첩 후보

In [ ]:
other_overlap_review = (
    overlap_candidates.loc[
        overlap_candidates["same_bbox"].eq(False)
    ]
    .sort_values(
        ["high_iou", "small_box_overlap", "iou"],
        ascending=False,
    )
    .reset_index(drop=True)
    .copy()
)

other_overlap_review.insert(
    0,
    "review_number",
    range(1, len(other_overlap_review) + 1),
)

print(f"나머지 중첩 검토 대상: {len(other_overlap_review):,}장")
print("※ 이번 Review 번호는 앞서 확인한 67장과 별도로 시작합니다.")

# 비교 지표 확인
display(other_overlap_review[[
    "review_number",
    "annotation_id_a",
    "drug_code_a",
    "annotation_id_b",
    "drug_code_b",
    "iou",
    "small_box_overlap",
]])

# 3장씩 전체 페이지 출력
page_size = 3
total_pages = (
    len(other_overlap_review) + page_size - 1
) // page_size

for page in range(1, total_pages + 1):
    show_same_bbox_page(
        page=page,
        page_size=page_size,
        review_table=other_overlap_review,
    )

### 동일 bbox 외 중첩 후보 시각 검토 결과

#### 검토 대상
- 동일 bbox를 제외한 중첩 후보 27장을 시각 검토했다.
- 첫 3장: bbox가 완전히 같지는 않지만 IoU가 0.8 이상인 후보.
- 나머지 24장: IoU는 0.8 미만이지만,
  작은 박스 면적 중 겹친 비율이 90% 이상인 후보.

#### 확인 결과 및 판단
- 나머지 24장에서는 박스가 없는 알약의 annotation이
  다른 알약 위치에 잘못 배치된 것을 확인했다.
- 첫 3장은 IoU 기준으로 검출된 별도 묶음이며,
  위 24장과 동일한 오류 유형으로 일괄 기록하지 않는다.
- 위치 오류 수정 시 annotation의 클래스와 실제 알약을 대조한 뒤,
  bbox의 위치·크기를 올바른 알약에 맞게 변경한다.
- 클래스 정보와 annotation ID는 유지하고 area를 재계산한다.
- 실제 수정은 아직 수행하지 않았다.

#### 다음 작업
- 동일 bbox 67장과 위치 오류가 확인된 24장을 우선 수정 대상으로 관리한다.
- 첫 3장은 개별 관찰 결과에 따라 수정 여부를 정한다.
- 수정 대상 한 장의 클래스·위치를 확인하고 수정 전·후를 비교한다.

### 2-5. bbox 위치 보정

In [ ]:
import json

# 동일 bbox 검토 1번을 첫 보정 대상으로 선택
repair_candidate = same_bbox_review.iloc[0].copy()
repair_json_path = SELECT_DIR / repair_candidate["json_file"]

with repair_json_path.open(encoding="utf-8-sig") as f:
    repair_document = json.load(f)

print("========== 첫 번째 보정 대상 ==========")
print(f"JSON: {repair_candidate['json_file']}")
print(
    f"A: ID {repair_candidate['annotation_id_a']} / "
    f"{repair_candidate['drug_code_a']}"
)
print(
    f"B: ID {repair_candidate['annotation_id_b']} / "
    f"{repair_candidate['drug_code_b']}"
)

print("\n========== 전체 annotation ==========")
display(pd.DataFrame(repair_document["annotations"])[[
    "id",
    "image_id",
    "drug_code",
    "category_id",
    "model_class_id",
    "source_dl_idx",
    "bbox",
    "area",
]])

print("\n========== 클래스 정보 ==========")
print(json.dumps(
    repair_document["categories"],
    ensure_ascii=False,
    indent=2,
))

print("\n========== 원본 출처 ==========")
print(json.dumps(
    repair_document["source"],
    ensure_ascii=False,
    indent=2,
))

print("\n※ 아직 좌표나 파일을 수정하지 않았습니다.")

In [ ]:
# 추가 정보 확인을 위한 원본 ZIP 내 JSON 검색 

import json
import zipfile
from pathlib import Path

AIHUB_DIR = SELECT_DIR.parent / "aihub"
original_zip_path = AIHUB_DIR / "TL_1_조합.zip"

target_image_name = repair_document["source"]["original_image"]

if not original_zip_path.is_file():
    raise FileNotFoundError(
        f"원본 라벨 ZIP을 찾을 수 없습니다: {original_zip_path}"
    )

original_matches = []
original_read_errors = []

# 원본은 알약별 JSON으로 나뉘어 있을 수 있으므로
# 같은 이미지를 참조하는 JSON을 모두 찾음
with zipfile.ZipFile(original_zip_path, "r") as z:
    json_members = [
        info for info in z.infolist()
        if not info.is_dir()
        and info.filename.lower().endswith(".json")
        and not info.filename.startswith("__MACOSX/")
    ]

    for member in json_members:
        try:
            data = json.loads(
                z.read(member).decode("utf-8-sig")
            )
        except (ValueError, OSError, UnicodeError) as error:
            original_read_errors.append({
                "member": member.filename,
                "error": str(error),
            })
            continue

        if not isinstance(data, dict):
            continue

        image_records = data.get("images", [])
        if not isinstance(image_records, list):
            continue

        matches_image = any(
            isinstance(record, dict)
            and any(
                isinstance(record.get(field), str)
                and Path(record[field]).name == target_image_name
                for field in ["file_name", "imgfile"]
            )
            for record in image_records
        )

        if matches_image:
            original_matches.append({
                "member": member.filename,
                "data": data,
            })

print(f"검사 ZIP: {original_zip_path}")
print(f"대상 이미지: {target_image_name}")
print(f"일치하는 원본 JSON: {len(original_matches)}개")
print(f"원본 읽기 실패: {len(original_read_errors)}개")

for index, match in enumerate(original_matches, start=1):
    print(f"\n========== 원본 JSON {index} ==========")
    print(f"ZIP 내부 경로: {match['member']}")
    print(json.dumps(
        match["data"],
        ensure_ascii=False,
        indent=2,
    ))

if not original_matches:
    print("\n일치하는 원본을 찾지 못했습니다. 검색 조건 확인이 필요합니다.")

if original_read_errors:
    display(pd.DataFrame(original_read_errors).head(10))

print("\n※ 압축 해제 및 파일 수정은 수행하지 않았습니다.")

In [ ]:
# ID 76(K-012778, 다타민큐정)의 박스를 왼쪽 위 붉은 알약으로 옮긴 미리보기

from copy import deepcopy

# 1. 수정 대상과 초기 좌표
target_annotation_id = 76
target_drug_code = "K-012778"

# [왼쪽 x, 위쪽 y, 너비, 높이]
# 이미지에서 추정한 초기값 — 미리보기 확인 후 조정
proposed_bbox = [190, 120, 210, 470]

# 디스크의 현재 JSON을 읽고, 미리보기용 복사본만 수정
with repair_json_path.open(encoding="utf-8-sig") as f:
    repair_before_document = json.load(f)

repair_preview_document = deepcopy(repair_before_document)

targets = [
    ann for ann in repair_preview_document["annotations"]
    if ann["id"] == target_annotation_id
]

if len(targets) != 1:
    raise ValueError("수정 대상 annotation을 하나로 확정할 수 없습니다.")

target = targets[0]

if target["drug_code"] != target_drug_code:
    raise ValueError("대상 ID와 drug_code가 예상한 조합과 다릅니다.")

image_records = [
    record for record in repair_preview_document["images"]
    if record["id"] == target["image_id"]
]

if len(image_records) != 1:
    raise ValueError("대상 annotation의 이미지 정보를 확정할 수 없습니다.")

image_info = image_records[0]
image_path = resolve_review_image(image_info["file_name"])

with Image.open(image_path) as image:
    preview_image = image.convert("RGB")

if preview_image.size != (
    image_info["width"], image_info["height"]
):
    raise ValueError("실제 이미지 크기와 JSON 크기가 다릅니다.")

x, y, w, h = proposed_bbox

if not (
    x >= 0 and y >= 0 and w > 0 and h > 0
    and x + w <= preview_image.width
    and y + h <= preview_image.height
):
    raise ValueError("수정 좌표가 이미지 범위를 벗어나거나 크기가 잘못됐습니다.")

before_bbox = target["bbox"].copy()
before_area = target["area"]

target["bbox"] = list(proposed_bbox)
target["area"] = w * h

# 2. 수정 전·후 비교
fig, axes = plt.subplots(1, 2, figsize=(14, 9))

for ax, document, title, color in [
    (axes[0], repair_before_document, "Before", "red"),
    (axes[1], repair_preview_document, "After (preview)", "blue"),
]:
    ax.imshow(preview_image)

    # 다른 annotation을 먼저 그리고 수정 대상을 마지막에 강조
    annotations = sorted(
        document["annotations"],
        key=lambda ann: ann["id"] == target_annotation_id,
    )

    for ann in annotations:
        bx, by, bw, bh = ann["bbox"]
        is_target = ann["id"] == target_annotation_id

        ax.add_patch(Rectangle(
            (bx, by), bw, bh,
            fill=False,
            edgecolor=color if is_target else "gray",
            linewidth=2.5 if is_target else 1,
        ))

        # 수정 대상은 위쪽, 나머지는 아래쪽에 표시
        ax.text(
            bx,
            by if is_target else by + bh,
            f"ID {ann['id']} | {ann['drug_code']}",
            fontsize=8,
            color=color if is_target else "black",
            va="bottom" if is_target else "top",
            bbox=dict(
                facecolor="white", alpha=0.8, edgecolor="none"
            ),
        )

    ax.set_title(
        f"{title}\nID {target_annotation_id} | {target_drug_code}"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

display(pd.DataFrame([
    {"상태": "수정 전", "bbox": before_bbox, "area": before_area},
    {
        "상태": "수정 후 미리보기",
        "bbox": target["bbox"],
        "area": target["area"],
    },
]))

print("※ 미리보기용 복사본만 변경했습니다. JSON 파일은 아직 저장하지 않았습니다.")

In [ ]:
import os
import tempfile
from datetime import datetime
from copy import deepcopy


def save_bbox_correction(json_path, before, after, annotation_id):
    """미리보기와 대조 후 백업하고 bbox·area만 저장."""
    with json_path.open(encoding="utf-8-sig") as f:
        current = json.load(f)

    # 저장 셀을 다시 실행한 경우
    if current == after:
        print("이미 동일한 수정 내용이 저장되어 있습니다.")
        return

    if current != before:
        raise ValueError(
            "미리보기 이후 파일 내용이 달라졌습니다. 다시 확인해 주세요."
        )

    before_targets = [
        ann for ann in before["annotations"]
        if ann["id"] == annotation_id
    ]
    after_targets = [
        ann for ann in after["annotations"]
        if ann["id"] == annotation_id
    ]

    if len(before_targets) != 1 or len(after_targets) != 1:
        raise ValueError("수정 대상을 하나로 확정할 수 없습니다.")

    # 다른 필드까지 변경되지 않았는지 확인
    expected = deepcopy(before)
    expected_target = next(
        ann for ann in expected["annotations"]
        if ann["id"] == annotation_id
    )
    expected_target["bbox"] = after_targets[0]["bbox"]
    expected_target["area"] = after_targets[0]["area"]

    if expected != after:
        raise ValueError("대상 bbox·area 외에 변경된 내용이 있습니다.")

    # 백업은 검사 대상인 select 밖에 보관
    backup_dir = (
        SELECT_DIR.parent
        / "bbox_backups"
        / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    )
    backup_dir.mkdir(parents=True, exist_ok=False)
    backup_path = backup_dir / json_path.name
    backup_path.write_bytes(json_path.read_bytes())

    # 임시 파일을 완성한 뒤 대상 파일을 교체
    temporary_path = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w",
            encoding="utf-8",
            dir=json_path.parent,
            suffix=".tmp",
            delete=False,
        ) as f:
            temporary_path = Path(f.name)
            json.dump(after, f, ensure_ascii=False, indent=2)
            f.write("\n")

        os.replace(temporary_path, json_path)
    finally:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)

    with json_path.open(encoding="utf-8-sig") as f:
        saved = json.load(f)

    if saved != after:
        raise RuntimeError(f"저장 결과 확인이 필요합니다. 백업: {backup_path}")

    print(f"저장 확인 완료: {json_path.name}")
    print(f"수정 annotation ID: {annotation_id}")
    print(f"bbox: {after_targets[0]['bbox']}")
    print(f"area: {after_targets[0]['area']}")
    print(f"수정 전 백업: {backup_path}")


# 위에서 정의한 함수를 호출해 이번 보정 저장
save_bbox_correction(
    json_path=repair_json_path,
    before=repair_before_document,
    after=repair_preview_document,
    annotation_id=76,
)

In [ ]:
# 1. 현재 JSON에서 아직 동일 bbox가 남은 대상만 구성
coordinate_search_targets = []

for _, candidate in same_bbox_review.iterrows():
    json_file = candidate["json_file"]

    with (SELECT_DIR / json_file).open(encoding="utf-8-sig") as f:
        document = json.load(f)

    ann_lookup = {
        ann["id"]: ann for ann in document["annotations"]
    }
    ann_a = ann_lookup[candidate["annotation_id_a"]]
    ann_b = ann_lookup[candidate["annotation_id_b"]]

    # 저장된 보정이 반영되어 두 박스가 달라졌으면 제외
    if ann_a["bbox"] != ann_b["bbox"]:
        continue

    image_lookup_local = {
        image["id"]: image for image in document["images"]
    }

    for ann in [ann_a, ann_b]:
        image = image_lookup_local[ann["image_id"]]

        coordinate_search_targets.append({
            "json_file": json_file,
            "annotation_id": ann["id"],
            "drug_code": ann["drug_code"],
            "original_image": Path(
                document["source"]["original_image"]
            ).name,
            "width": image["width"],
            "height": image["height"],
            "current_bbox": list(ann["bbox"]),
        })

targets_by_image = {}
for target in coordinate_search_targets:
    targets_by_image.setdefault(
        target["original_image"], []
    ).append(target)

target_json_names = {
    Path(name).with_suffix(".json").name
    for name in targets_by_image
}

# 2. 全 조합 라벨 ZIP에서 같은 이미지·같은 클래스 검색
aihub_dir = SELECT_DIR.parent / "aihub"
label_archives = sorted(aihub_dir.glob("TL_*_조합.zip"))

if not label_archives:
    raise FileNotFoundError(f"조합 라벨 ZIP을 찾지 못했습니다: {aihub_dir}")

different_bbox_rows = []
coordinate_search_errors = []
matched_target_keys = set()
same_bbox_record_count = 0

for archive_path in label_archives:
    print(f"검색 중: {archive_path.name}")

    with zipfile.ZipFile(archive_path, "r") as z:
        for member in z.infolist():
            if (
                member.is_dir()
                or member.filename.startswith("__MACOSX/")
                or Path(member.filename).name not in target_json_names
            ):
                continue

            try:
                original = json.loads(
                    z.read(member).decode("utf-8-sig")
                )

                for image in original["images"]:
                    names = {
                        Path(image[field]).name
                        for field in ["file_name", "imgfile"]
                        if isinstance(image.get(field), str)
                    }

                    matching_names = names & targets_by_image.keys()
                    code = (
                        image.get("dl_mapping_code")
                        or image.get("drug_N")
                    )

                    for image_name in matching_names:
                        for target in targets_by_image[image_name]:
                            if code != target["drug_code"]:
                                continue

                            for ann in original["annotations"]:
                                if ann.get("image_id") != image.get("id"):
                                    continue

                                bbox = ann.get("bbox")
                                if not (
                                    isinstance(bbox, list)
                                    and len(bbox) == 4
                                    and all(
                                        is_finite_number(v) for v in bbox
                                    )
                                ):
                                    continue

                                x, y, w, h = bbox
                                if not (
                                    image.get("width") == target["width"]
                                    and image.get("height") == target["height"]
                                    and x >= 0 and y >= 0
                                    and w > 0 and h > 0
                                    and x + w <= target["width"]
                                    and y + h <= target["height"]
                                ):
                                    continue

                                matched_target_keys.add((
                                    target["json_file"],
                                    target["annotation_id"],
                                ))

                                if bbox == target["current_bbox"]:
                                    same_bbox_record_count += 1
                                    continue

                                different_bbox_rows.append({
                                    **target,
                                    "candidate_bbox": bbox,
                                    "product_name": image.get("dl_name"),
                                    "print_front": image.get("print_front"),
                                    "print_back": image.get("print_back"),
                                    "source_zip": archive_path.name,
                                    "source_member": member.filename,
                                })

            except (
                ValueError, KeyError, TypeError, OSError
            ) as error:
                coordinate_search_errors.append({
                    "zip": archive_path.name,
                    "member": member.filename,
                    "error": str(error),
                })

# 3. 검색 결과 유지
different_bbox_candidates = pd.DataFrame(
    different_bbox_rows,
    columns=[
        "json_file", "annotation_id", "drug_code",
        "original_image", "width", "height", "current_bbox",
        "candidate_bbox", "product_name", "print_front",
        "print_back", "source_zip", "source_member",
    ],
)

target_image_count = len({
    target["json_file"] for target in coordinate_search_targets
})

print("\n========== 원본 좌표 대조 결과 ==========")
print(f"검색한 라벨 ZIP: {len(label_archives)}개")
print(f"아직 동일 bbox가 남은 이미지: {target_image_count}장")
print(f"검색 대상 annotation: {len(coordinate_search_targets)}개")
print(f"유효한 원본 기록을 찾은 annotation: {len(matched_target_keys)}개")
print(f"현재와 동일한 좌표 기록: {same_bbox_record_count}건")
print(f"현재와 다른 좌표 기록: {len(different_bbox_candidates)}건")
print(
    "다른 좌표 후보가 발견된 이미지: "
    f"{different_bbox_candidates['json_file'].nunique()}장"
)
print(f"읽기·구조 확인 중 오류: {len(coordinate_search_errors)}건")

if different_bbox_candidates.empty:
    print("\n검색한 원본에서 대체 좌표 후보를 찾지 못했습니다.")
else:
    display(different_bbox_candidates[[
        "json_file", "annotation_id", "drug_code",
        "product_name", "current_bbox", "candidate_bbox", "source_zip",
    ]].head(30))

if coordinate_search_errors:
    display(pd.DataFrame(coordinate_search_errors).head(10))

print("\n※ 파일 수정 없음")
print("※ 다른 좌표는 정상 판정 전의 후보입니다.")
print("※ 다른 ZIP에서 찾은 같은 파일명은 실제 이미지 동일성도 확인해야 합니다.")

### bbox 중복·중첩 검토 및 최종 처리

- 검토 대상: 총 94장
  - 동일 bbox: 67장
  - 동일 bbox 외 IoU 0.8 이상: 3장
  - 위 항목 외 작은 박스 중첩 비율 90% 이상: 24장
- 시각 검토에서 한 알약에 복수 annotation이 배치되거나,
  bbox의 위치·크기가 실제 알약과 맞지 않는 사례를 확인했다.
- 동일 bbox 중 미보정 66장의 원본 annotation 132개를
  전체 조합 라벨 ZIP 7개와 대조했으나 대체 좌표는 없었다.
- 나머지 27장도 현재 JSON 정보만으로 올바른 위치를
  확정할 수 없어, 추가 보정 없이 제외하기로 결정했다.
- 첫 1장의 시험 보정은 완료했으나 최종 제외 대상에 포함한다.(AI를 활용한 보정)
- 최종 결정: 94장 모두 학습 대상에서 제외한다.
- 원본 파일은 보존하고 학습 데이터 구성에서 제외 목록을 적용한다.

### 2-6. 실제 이미지 파일 누락·손상·크기 검사

| 점검표 항목 | 이번 확인 내용 |
|---|---|
| 이미지 누락 | JSON이 가리키는 파일이 존재하는지 |
| 이미지 손상 | 파일 검사와 실제 픽셀 읽기가 가능한지 |
| 이미지 크기 불일치 | 실제 너비·높이가 JSON과 같은지 |
| 연결되지 않은 이미지 | 이미지 파일은 있지만 JSON에서 참조하지 않는지 |

In [ ]:
image_file_issue_rows = []
referenced_image_paths = set()
image_read_success_count = 0


def add_image_file_issue(row, item, detail):
    image_file_issue_rows.append({
        "json_file": row["_json_file"],
        "image_id": row["id"],
        "file_name": row["file_name"],
        "check_item": item,
        "detail": detail,
    })


for number, (_, row) in enumerate(images_df.iterrows(), start=1):
    # 1. 실제 파일 연결
    try:
        image_path = resolve_review_image(row["file_name"])
        referenced_image_paths.add(image_path.resolve())
    except (ValueError, OSError) as error:
        add_image_file_issue(
            row, "이미지 누락 또는 경로 불명확", str(error)
        )
        continue

    # 2. 파일 구조 검사 + 실제 픽셀 읽기
    try:
        with Image.open(image_path) as image:
            image.verify()

        # verify 이후에는 다시 열어야 함
        with Image.open(image_path) as image:
            image.load()
            actual_width, actual_height = image.size

        image_read_success_count += 1

    except Exception as error:
        add_image_file_issue(
            row,
            "이미지 읽기 실패",
            f"{type(error).__name__}: {error}",
        )
        continue

    # 3. 실제 이미지 크기와 JSON 크기 비교
    if (
        actual_width != row["width"]
        or actual_height != row["height"]
    ):
        add_image_file_issue(
            row,
            "이미지 크기 불일치",
            f"실제={actual_width}×{actual_height}, "
            f"JSON={row['width']}×{row['height']}",
        )

    if number % 1000 == 0:
        print(f"진행: {number:,}/{len(images_df):,}개")


image_file_issues = pd.DataFrame(
    image_file_issue_rows,
    columns=[
        "json_file", "image_id", "file_name",
        "check_item", "detail",
    ],
)

# 4. 실제 파일 중 JSON에서 참조하지 않는 이미지
# 앞서 구성한 select 이미지 경로 조회표 재사용
all_actual_image_paths = {
    path.resolve()
    for paths in review_image_paths.values()
    for path in paths
}

unreferenced_image_paths = sorted(
    all_actual_image_paths - referenced_image_paths
)

print("\n========== 실제 이미지 검사 ==========")
print(f"검사 이미지 기록: {len(images_df):,}개")
print(f"픽셀 읽기 성공: {image_read_success_count:,}개")
print(f"문제 기록: {len(image_file_issues):,}건")
print(f"JSON에서 참조하지 않는 이미지: {len(unreferenced_image_paths):,}개")

if image_file_issues.empty:
    print("\n[체크 가능] 이미지 누락 — 없음")
    print("[체크 가능] 이미지 손상 — 이번 읽기 검사에서 없음")
    print("[체크 가능] 실제 이미지·JSON 크기 불일치 — 없음")
else:
    display(
        image_file_issues.groupby("check_item")
        .agg(
            문제기록수=("json_file", "size"),
            영향JSON수=("json_file", "nunique"),
        )
        .reset_index()
    )
    display(image_file_issues.head(20))

if unreferenced_image_paths:
    print("\n========== 참조되지 않은 이미지: 최대 20개 ==========")
    for path in unreferenced_image_paths[:20]:
        print(path.relative_to(SELECT_DIR.resolve()))
else:
    print("[체크 가능] JSON에서 참조하지 않는 이미지 — 없음")

print("\n※ 이미지 내용과 클래스의 일치 여부는 이번 검사 대상이 아닙니다.")

### 2-7. ignore / iscrowd 값 검사

In [ ]:
# 2-7. ignore / iscrowd 값 검사
# 필드 존재 여부를 보존한 2-1의 원래 기록 재사용

flag_check_rows = []

for annotation in table_rows["annotations"]:
    for field in ["ignore", "iscrowd"]:
        value = annotation.get(field)

        if field not in annotation:
            status = "필드 없음"
        elif not is_integer_number(value):
            status = "값 형식 오류"
        elif value == 0:
            status = "정상: 0"
        else:
            status = "의심: 0 이외"

        flag_check_rows.append({
            "json_file": annotation["_json_file"],
            "annotation_id": annotation["id"],
            "drug_code": annotation.get("drug_code"),
            "field": field,
            "value": value,
            "status": status,
        })

flag_checks = pd.DataFrame(flag_check_rows)

flag_summary = (
    flag_checks.groupby(["field", "status"])
    .agg(
        annotation_count=("annotation_id", "size"),
        json_count=("json_file", "nunique"),
    )
    .reset_index()
)

flag_issues = flag_checks.loc[
    flag_checks["status"].isin([
        "값 형식 오류",
        "의심: 0 이외",
    ])
].copy()

flag_missing = flag_checks.loc[
    flag_checks["status"].eq("필드 없음")
].copy()

print("========== 2-7. ignore / iscrowd 검사 ==========")
print(f"검사 annotation: {len(table_rows['annotations']):,}개")
display(flag_summary.rename(columns={
    "field": "필드",
    "status": "결과",
    "annotation_count": "Annotation 수",
    "json_count": "JSON 수",
}))

if not flag_issues.empty:
    print("\n========== 의심 값: 최대 20건 ==========")
    display(flag_issues.head(20))
else:
    print("\n존재하는 필드에서 0 이외의 값·형식 오류 없음")

if not flag_missing.empty:
    print("\n※ 누락 필드는 정상 0으로 판정하지 않았습니다.")
    print("※ 선정본 생성 시 생략됐는지 원본 정보와 함께 확인해야 합니다.")
else:
    if flag_issues.empty:
        print("[체크 가능] ignore / iscrowd 값 이상 — 없음")

print("\n※ 파일 수정·삭제 없음")

---
---
### 3-1. 중복 또는 유사 이미지 검사

In [ ]:
# 3-1. 중복 또는 유사 이미지 검사
# 전체 select 검사 — 파일 수정·삭제 없음

import numpy as np
import hashlib

PHASH_DISTANCE_THRESHOLD = 5

# --------------------------------------------------
# 1. 해시 계산 함수 준비
# --------------------------------------------------

dct_size = 32
positions = np.arange(dct_size)
frequencies = positions[:, None]

dct_matrix = np.cos(
    np.pi * (2 * positions + 1) * frequencies / (2 * dct_size)
)
dct_matrix *= np.sqrt(2 / dct_size)
dct_matrix[0] /= np.sqrt(2)


def calculate_image_hashes(image_path):
    """RGB 픽셀의 MD5와 64비트 pHash 계산."""
    with Image.open(image_path) as image:
        rgb = image.convert("RGB")

        # 파일 압축 방식이 아닌 실제 픽셀 내용 비교
        digest = hashlib.md5()
        digest.update(
            f"RGB:{rgb.width}:{rgb.height}:".encode("utf-8")
        )
        digest.update(rgb.tobytes())

        gray = np.asarray(
            rgb.convert("L").resize(
                (dct_size, dct_size),
                Image.Resampling.LANCZOS,
            ),
            dtype=np.float64,
        )

    coefficients = dct_matrix @ gray @ dct_matrix.T
    low_frequency = coefficients[:8, :8]
    bits = (
        low_frequency > np.median(low_frequency)
    ).ravel()

    phash = 0
    for bit in bits:
        phash = (phash << 1) | int(bit)

    return digest.hexdigest(), phash


# --------------------------------------------------
# 2. 전체 이미지 해시 계산
# --------------------------------------------------

source_lookup = sources_df.set_index("_json_file")
image_hash_rows = []
image_hash_error_rows = []

for number, (_, row) in enumerate(images_df.iterrows(), start=1):
    try:
        original_name = Path(
            source_lookup.loc[
                row["_json_file"], "original_image"
            ]
        ).name

        # 예: K-000250-000573-005002-012778
        combination_codes = (
            original_name.split("_")[0].split("-")
        )

        if (
            combination_codes[0] != "K"
            or len(combination_codes) < 2
            or not all(
                code.isdigit() for code in combination_codes[1:]
            )
        ):
            raise ValueError(
                f"원본 파일명에서 조합을 읽을 수 없음: {original_name}"
            )

        combination = tuple(sorted(
            int(code) for code in combination_codes[1:]
        ))

        image_path = resolve_review_image(row["file_name"])
        pixel_md5, phash = calculate_image_hashes(image_path)

        image_hash_rows.append({
            "json_file": row["_json_file"],
            "image_id": row["id"],
            "file_name": row["file_name"],
            "combination": combination,
            "pixel_md5": pixel_md5,
            "phash": phash,
        })

    except Exception as error:
        image_hash_error_rows.append({
            "json_file": row["_json_file"],
            "error": f"{type(error).__name__}: {error}",
        })

    if number % 1000 == 0 or number == len(images_df):
        print(
            f"해시 계산: {number:,}/{len(images_df):,}장",
            flush=True,
        )

image_hashes = pd.DataFrame(image_hash_rows)

image_hash_errors = pd.DataFrame(
    image_hash_error_rows,
    columns=["json_file", "error"],
)

if image_hashes.empty:
    raise ValueError("해시 계산에 성공한 이미지가 없습니다.")


# --------------------------------------------------
# 3. 완전 중복 검사
# 같은 조합도 포함하여 RGB 픽셀이 동일한 이미지 검색
# --------------------------------------------------

exact_duplicate_images = image_hashes.loc[
    image_hashes["pixel_md5"].duplicated(keep=False)
].copy()

exact_duplicate_group_count = (
    exact_duplicate_images["pixel_md5"].nunique()
)

exact_duplicate_excess_count = (
    len(exact_duplicate_images) - exact_duplicate_group_count
)


# --------------------------------------------------
# 4. 유사 이미지 검사
# 서로 다른 조합이고 pHash 해밍 거리가 5 이하인 쌍
# 완전 중복은 위에서 별도 집계하므로 여기서는 제외
# --------------------------------------------------

# 64비트를 6개 구간으로 나눠 비교 대상을 줄임.
# 차이가 5비트 이하면 적어도 한 구간은 같으므로,
# 설정한 거리 조건을 만족하는 쌍을 놓치지 않음.
block_widths = [11, 11, 11, 11, 10, 10]
block_indexes = [{} for _ in block_widths]

hash_records = image_hashes.to_dict("records")
similar_image_rows = []
compared_pair_count = 0

for index, record in enumerate(hash_records):
    current_hash = int(record["phash"])
    possible_matches = set()
    block_keys = []
    shift = 0

    for block_number, width in enumerate(block_widths):
        key = (current_hash >> shift) & ((1 << width) - 1)
        block_keys.append(key)

        possible_matches.update(
            block_indexes[block_number].get(key, [])
        )
        shift += width

    for previous_index in possible_matches:
        previous = hash_records[previous_index]

        # 같은 조합의 각도 차이 등은 유사 후보에서 제외
        if previous["combination"] == record["combination"]:
            continue

        # 완전 중복은 별도 결과에 보관
        if previous["pixel_md5"] == record["pixel_md5"]:
            continue

        compared_pair_count += 1
        distance = (
            int(previous["phash"]) ^ current_hash
        ).bit_count()

        if distance <= PHASH_DISTANCE_THRESHOLD:
            similar_image_rows.append({
                "json_file_a": previous["json_file"],
                "image_id_a": previous["image_id"],
                "file_name_a": previous["file_name"],
                "combination_a": previous["combination"],
                "json_file_b": record["json_file"],
                "image_id_b": record["image_id"],
                "file_name_b": record["file_name"],
                "combination_b": record["combination"],
                "phash_distance": distance,
            })

    for block_number, key in enumerate(block_keys):
        block_indexes[block_number].setdefault(
            key, []
        ).append(index)

    if (
        (index + 1) % 1000 == 0
        or index + 1 == len(hash_records)
    ):
        print(
            f"유사 비교: {index + 1:,}/{len(hash_records):,}장",
            flush=True,
        )

similar_image_candidates = pd.DataFrame(
    similar_image_rows,
    columns=[
        "json_file_a",
        "image_id_a",
        "file_name_a",
        "combination_a",
        "json_file_b",
        "image_id_b",
        "file_name_b",
        "combination_b",
        "phash_distance",
    ],
).sort_values(
    "phash_distance",
    ignore_index=True,
)

similar_image_ids = (
    set(similar_image_candidates["image_id_a"])
    | set(similar_image_candidates["image_id_b"])
)


# --------------------------------------------------
# 5. 결과 출력
# --------------------------------------------------

print("\n========== 3-1. 중복·유사 이미지 검사 ==========")
print(f"전체 이미지 기록: {len(images_df):,}개")
print(f"해시 계산 성공: {len(image_hashes):,}개")
print(f"해시 계산 실패: {len(image_hash_errors):,}개")

print("\n[완전 중복 — 같은 조합 포함]")
print(f"완전 중복 그룹: {exact_duplicate_group_count:,}개")
print(f"중복 그룹에 포함된 이미지: {len(exact_duplicate_images):,}장")
print(
    f"그룹당 1장 유지 시 중복 초과분: "
    f"{exact_duplicate_excess_count:,}장"
)

print("\n[유사 이미지 — 서로 다른 조합만]")
print(f"pHash 해밍 거리 기준: {PHASH_DISTANCE_THRESHOLD} 이하")
print(f"해밍 거리를 계산한 비교 쌍: {compared_pair_count:,}쌍")
print(f"유사 후보: {len(similar_image_candidates):,}쌍")
print(f"유사 후보에 포함된 이미지: {len(similar_image_ids):,}장")

if not exact_duplicate_images.empty:
    print("\n========== 완전 중복: 최대 20개 기록 ==========")
    display(
        exact_duplicate_images.sort_values(
            ["pixel_md5", "file_name"]
        )[[
            "pixel_md5",
            "image_id",
            "file_name",
            "combination",
        ]].head(20)
    )

if not similar_image_candidates.empty:
    print("\n========== 유사 후보 거리 분포 ==========")
    display(
        similar_image_candidates.groupby("phash_distance")
        .size()
        .rename("후보쌍수")
        .reset_index()
    )

    print("\n========== 유사 후보: 최대 20쌍 ==========")
    display(similar_image_candidates.head(20))

if not image_hash_errors.empty:
    print("\n========== 해시 계산 실패: 최대 20건 ==========")
    display(image_hash_errors.head(20))

if image_hash_errors.empty:
    if (
        exact_duplicate_images.empty
        and similar_image_candidates.empty
    ):
        print("\n[체크 가능] 설정한 기준에서 중복·유사 후보 없음")
    else:
        print("\n자동 검사 완료 — 발견된 후보의 후속 검토 필요")
else:
    print("\n검사 실패 이미지가 있어 전체 검사 완료 판정은 보류")

print("\n※ pHash가 같아도 실제 픽셀이 같다는 뜻은 아닙니다.")
print("※ 유사 후보는 오류나 제외 대상으로 확정한 결과가 아닙니다.")
print("※ 제외 예정 94장을 포함한 전체 이미지 검사입니다.")
print("※ 파일 수정·삭제 없음")

In [ ]:
# pHash 거리 0·2·4별 최대 3쌍, 총 9쌍을 고정된 난수로 뽑아 표시

def show_similarity_samples(candidates, per_distance=3, seed=42):
    """pHash 거리별 표본을 추출해 두 이미지를 나란히 표시."""
    if candidates.empty:
        print("시각 검토할 유사 후보가 없습니다.")
        return candidates.copy()

    samples = pd.concat(
        [
            group.sample(
                n=min(per_distance, len(group)),
                random_state=seed,
            )
            for _, group in candidates.groupby(
                "phash_distance", sort=True
            )
        ],
        ignore_index=True,
    )
    samples.insert(0, "review_number", range(1, len(samples) + 1))

    for _, pair in samples.iterrows():
        print(
            f"\n검토 {pair['review_number']} | "
            f"pHash 거리 {pair['phash_distance']}"
        )
        print(f"A: {pair['file_name_a']}")
        print(f"B: {pair['file_name_b']}")

        codes_a = set(pair["combination_a"])
        codes_b = set(pair["combination_b"])

        print(
            "A에만 있는 클래스:",
            ", ".join(f"K-{code:06d}" for code in sorted(codes_a - codes_b)),
        )
        print(
            "B에만 있는 클래스:",
            ", ".join(f"K-{code:06d}" for code in sorted(codes_b - codes_a)),
        )

        fig, axes = plt.subplots(1, 2, figsize=(12, 8))

        for ax, side in zip(axes, ["a", "b"]):
            image_path = resolve_review_image(
                pair[f"file_name_{side}"]
            )
            with Image.open(image_path) as image:
                ax.imshow(image.convert("RGB"))

            ax.set_title(
                f"{side.upper()} | Image ID {pair[f'image_id_{side}']}"
            )
            ax.axis("off")

        plt.tight_layout()
        plt.show()

    return samples


# 거리별 표본 추출 및 시각화
similarity_review_samples = show_similarity_samples(
    similar_image_candidates,
    per_distance=3,
    seed=42,
)

### 중복·유사 이미지 검사 결과 및 판단

- 전체 이미지 10,240장의 해시 계산에 성공했으며 실패는 없었다.
- RGB 픽셀 MD5 기준 완전 중복은 없었다.
- 서로 다른 조합 중 pHash 해밍 거리 5 이하인 후보는
  10,168쌍이며, 해당 이미지는 5,227장이다.
- 거리 0·2·4별 3쌍씩 총 9쌍을 표본 시각 검토했다.
- 검토 표본에서는 공통 알약을 유지하면서 일부 알약을 교체한
  서로 다른 이미지임을 확인했다.
- pHash 유사도만으로 중복이나 라벨 오류로 판단하지 않으며,
  이번 검사로 추가 제외하지 않는다.
- 유사 후보 전체를 시각 검토한 것은 아니다.

### 3-2. 클래스 매핑 정합성 검사

In [ ]:
# 3-2. 클래스 매핑 정합성 검사
# 원본 매핑표 유지, 검사용 복사본의 자료형만 정리

checked_mapping = class_mapping.copy()

if (
    checked_mapping["code"].isna().any()
    or checked_mapping["code"].duplicated().any()
):
    raise ValueError("매핑표 code의 누락·중복을 확인해야 합니다.")

for field in ["category_id", "model_class_id", "aihub_dl_idx"]:
    # 숫자로 변환할 수 없는 문자열은 오류로 표시
    numeric = pd.to_numeric(
        checked_mapping[field], errors="raise"
    )

    # 결측이 아닌 값은 모두 유효한 정수여야 함
    if not numeric.dropna().map(is_integer_number).all():
        raise ValueError(f"{field}에 유효한 정수가 아닌 값이 있습니다.")

    # 결측을 유지할 수 있는 정수형
    checked_mapping[field] = numeric.astype("Int64")

    if checked_mapping[field].dropna().duplicated().any():
        raise ValueError(f"{field}에 중복 ID가 있습니다.")

mapping_lookup = checked_mapping.set_index("code")
mapping_issue_rows = []

field_pairs = [
    ("category_id", "category_id"),
    ("model_class_id", "model_class_id"),
    ("source_dl_idx", "aihub_dl_idx"),
]

for annotation in table_rows["annotations"]:
    code = annotation.get("drug_code")
    problems = []

    if not isinstance(code, str) or code not in mapping_lookup.index:
        problems.append((
            "매핑표에 없는 클래스",
            f"drug_code={code!r}",
        ))
    else:
        expected = mapping_lookup.loc[code]

        for annotation_field, mapping_field in field_pairs:
            actual_value = annotation.get(annotation_field)
            expected_value = expected[mapping_field]

            if pd.isna(expected_value):
                problems.append((
                    "사용 중인 클래스의 매핑값 누락",
                    f"매핑표 {mapping_field}가 결측값",
                ))
            elif not is_integer_number(actual_value):
                problems.append((
                    "annotation ID 형식 오류",
                    f"{annotation_field}={actual_value!r}",
                ))
            elif actual_value != int(expected_value):
                problems.append((
                    "클래스 매핑 불일치",
                    f"{annotation_field}={actual_value}, "
                    f"매핑표 {mapping_field}={int(expected_value)}",
                ))

    for item, detail in problems:
        mapping_issue_rows.append({
            "json_file": annotation["_json_file"],
            "annotation_id": annotation["id"],
            "drug_code": code,
            "check_item": item,
            "detail": detail,
        })

mapping_issues = pd.DataFrame(
    mapping_issue_rows,
    columns=[
        "json_file", "annotation_id", "drug_code",
        "check_item", "detail",
    ],
)

used_codes = set(annotations_df["drug_code"])
unused_missing_mapping = checked_mapping.loc[
    checked_mapping[
        ["category_id", "model_class_id", "aihub_dl_idx"]
    ].isna().any(axis=1)
    & ~checked_mapping["code"].isin(used_codes)
]

print("========== 3-2. 클래스 매핑 검사 ==========")
print(f"매핑표: {len(checked_mapping):,}행")
print(f"선정본 사용 클래스: {len(used_codes):,}종")
print(f"검사 annotation: {len(table_rows['annotations']):,}개")
print(f"문제 기록: {len(mapping_issues):,}건")
print(
    f"영향 JSON: "
    f"{mapping_issues['json_file'].nunique():,}개"
)

if mapping_issues.empty:
    print("\n[체크 가능] 선정본과 클래스 매핑표 간 불일치 — 없음")
else:
    display(
        mapping_issues.groupby("check_item")
        .agg(
            문제기록수=("annotation_id", "size"),
            영향JSON수=("json_file", "nunique"),
        )
        .reset_index()
    )
    display(mapping_issues.head(20))

if not unused_missing_mapping.empty:
    print("\n현재 미사용이므로 검사에 영향 없는 결측 클래스:")
    display(unused_missing_mapping[[
        "code", "category_id", "model_class_id", "aihub_dl_idx"
    ]])

print("\n※ 결측 ID를 임의로 채우지 않았습니다.")
print("※ 원본 매핑표와 파일은 변경하지 않았습니다.")
print("※ 향후 결측 클래스를 추가할 때는 원본 ID 확인이 필요합니다.")

### 3-3. 파일명과 annotation의 클래스 구성 비교

In [ ]:
# 3-3. 파일명과 annotation의 클래스 구성 비교
# 점검표:
# [교차 검증 / 의심] 파일 이름의 dl_idx와 Annotation category_id 구성 불일치

filename_source_lookup = sources_df.set_index("_json_file")

annotation_groups_by_json = {
    json_file: group
    for json_file, group in annotations_df.groupby("_json_file")
}

filename_composition_rows = []

for _, image in images_df.iterrows():
    json_file = image["_json_file"]
    original_name = Path(
        filename_source_lookup.loc[json_file, "original_image"]
    ).name

    combination_text = original_name.split("_")[0]
    parts = combination_text.split("-")

    # 예: K-000250-000573-005002-012778
    filename_format_ok = (
        len(parts) >= 2
        and parts[0] == "K"
        and all(part.isdigit() for part in parts[1:])
    )

    annotation_group = annotation_groups_by_json.get(json_file)

    actual_codes = (
        set()
        if annotation_group is None
        else set(
            int(value)
            for value in annotation_group["category_id"]
        )
    )

    if filename_format_ok:
        expected_codes = {int(part) for part in parts[1:]}
        missing_codes = sorted(expected_codes - actual_codes)
        extra_codes = sorted(actual_codes - expected_codes)
        composition_ok = not missing_codes and not extra_codes
    else:
        expected_codes = set()
        missing_codes = []
        extra_codes = []
        composition_ok = False

    filename_composition_rows.append({
        "json_file": json_file,
        "image_id": image["id"],
        "original_image": original_name,
        "combination": combination_text,
        "filename_format_ok": filename_format_ok,
        "expected_codes": sorted(expected_codes),
        "actual_codes": sorted(actual_codes),
        "missing_codes": missing_codes,
        "extra_codes": extra_codes,
        "annotation_count": (
            0 if annotation_group is None else len(annotation_group)
        ),
        "composition_ok": composition_ok,
    })

filename_composition_checks = pd.DataFrame(
    filename_composition_rows
)

filename_composition_issues = filename_composition_checks.loc[
    ~filename_composition_checks["composition_ok"]
].copy()

print("========== 3-3. 파일명·클래스 구성 검사 ==========")
print(f"검사 이미지: {len(filename_composition_checks):,}장")
print(
    f"파일명 형식 확인 불가: "
    f"{(~filename_composition_checks['filename_format_ok']).sum():,}장"
)
print(
    f"파일명에는 있지만 annotation에 없는 클래스가 있는 이미지: "
    f"{filename_composition_checks['missing_codes'].map(bool).sum():,}장"
)
print(
    f"annotation에만 있는 클래스가 있는 이미지: "
    f"{filename_composition_checks['extra_codes'].map(bool).sum():,}장"
)
print(f"문제 이미지 합계: {len(filename_composition_issues):,}장")

if filename_composition_issues.empty:
    print(
        "\n[체크 가능] 파일 이름의 dl_idx와 "
        "Annotation category_id 구성 불일치 — 없음"
    )
else:
    display(filename_composition_issues[[
        "json_file",
        "original_image",
        "filename_format_ok",
        "expected_codes",
        "actual_codes",
        "missing_codes",
        "extra_codes",
    ]].head(20))

print("\n※ missing_codes: 파일명에 있으나 annotation에 없는 클래스")
print("※ extra_codes: annotation에 있으나 파일명에 없는 클래스")
print("※ 클래스 구성 검사이며 실제 사진 속 알약의 일치 여부는 별도입니다.")
print("※ 같은 클래스의 annotation 반복 여부는 이번 집합 비교로 판단하지 않습니다.")
print("※ 파일 수정·삭제 없음")

### 3-4. 클래스별 제품 정보 일관성 검사

In [ ]:
# 3-4. 클래스별 제품 정보 일관성 검사
# 점검표:
# [교차 검증 / 의심] 동일 클래스에 여러 제품명·상충하는 외형 정보

# 1. 선정본 annotation과 원본 출처 연결
expected_product_links = (
    annotations_df[["_json_file", "drug_code"]]
    .merge(
        sources_df[[
            "_json_file", "source_number", "original_image"
        ]],
        on="_json_file",
        how="left",
        validate="many_to_one",
    )
    .drop_duplicates()
)

expected_product_links["source_number"] = (
    expected_product_links["source_number"].astype(int)
)
expected_product_links["original_image"] = (
    expected_product_links["original_image"]
    .map(lambda name: Path(name).name)
)

expected_product_keys = set(
    expected_product_links[[
        "source_number", "original_image", "drug_code"
    ]].itertuples(index=False, name=None)
)

# 後続の検査でも使う原本情報をまとめて取得
product_metadata_fields = [
    "dl_name",
    "drug_shape",
    "color_class1",
    "color_class2",
    "print_front",
    "print_back",
    "leng_long",
    "dl_idx",
    "drug_N",
    "dl_mapping_code",
]

product_metadata_rows = []
product_read_error_rows = []
aihub_dir = SELECT_DIR.parent / "aihub"

# 2. 原本ラベルから対象画像・クラスの情報だけ収集
for source_number, group in expected_product_links.groupby(
    "source_number", sort=True
):
    archive_path = aihub_dir / f"TL_{source_number}_조합.zip"
    target_json_names = {
        Path(name).with_suffix(".json").name
        for name in group["original_image"]
    }

    if not archive_path.is_file():
        product_read_error_rows.append({
            "zip": archive_path.name,
            "member": "",
            "error": "원본 ZIP 없음",
        })
        continue

    print(f"제품 정보 수집: {archive_path.name}", flush=True)

    with zipfile.ZipFile(archive_path, "r") as z:
        for member in z.infolist():
            if (
                member.is_dir()
                or member.filename.startswith("__MACOSX/")
                or Path(member.filename).name not in target_json_names
            ):
                continue

            try:
                data = json.loads(
                    z.read(member).decode("utf-8-sig")
                )

                for image in data["images"]:
                    original_image = Path(
                        image.get("file_name") or image["imgfile"]
                    ).name
                    code = (
                        image.get("dl_mapping_code")
                        or image.get("drug_N")
                    )
                    key = (int(source_number), original_image, code)

                    if key not in expected_product_keys:
                        continue

                    product_metadata_rows.append({
                        "source_number": int(source_number),
                        "original_image": original_image,
                        "drug_code": code,
                        "source_member": member.filename,
                        **{
                            field: image.get(field)
                            for field in product_metadata_fields
                        },
                    })

            except Exception as error:
                product_read_error_rows.append({
                    "zip": archive_path.name,
                    "member": member.filename,
                    "error": f"{type(error).__name__}: {error}",
                })

original_product_metadata = pd.DataFrame(
    product_metadata_rows,
    columns=[
        "source_number", "original_image",
        "drug_code", "source_member",
        *product_metadata_fields,
    ],
)

# 3. 必要な原本情報がすべて見つかったか確認
found_product_keys = set(
    original_product_metadata[[
        "source_number", "original_image", "drug_code"
    ]].itertuples(index=False, name=None)
)
missing_product_keys = expected_product_keys - found_product_keys

# 4. 同一クラス内の値の種類を確認
check_product_fields = {
    "dl_name": "제품명",
    "drug_shape": "모양",
    "color_class1": "색상",
    "print_front": "앞면 각인",
}

product_variation_rows = []
product_missing_rows = []

for code, group in original_product_metadata.groupby("drug_code"):
    for field, label in check_product_fields.items():
        # 空欄と欠損は別集計し、値の不一致には数えない
        values = group[field].map(
            lambda value: (
                "" if pd.isna(value) else str(value).strip()
            )
        )
        distinct_values = sorted(set(values) - {""})

        if len(distinct_values) > 1:
            product_variation_rows.append({
                "drug_code": code,
                "field": field,
                "항목": label,
                "값 종류 수": len(distinct_values),
                "확인된 값": distinct_values,
            })

        missing_count = int(values.eq("").sum())
        if missing_count:
            product_missing_rows.append({
                "drug_code": code,
                "field": field,
                "항목": label,
                "빈 값 기록 수": missing_count,
            })

product_variations = pd.DataFrame(
    product_variation_rows,
    columns=[
        "drug_code", "field", "항목", "값 종류 수", "확인된 값"
    ],
)
product_missing_values = pd.DataFrame(
    product_missing_rows,
    columns=["drug_code", "field", "항목", "빈 값 기록 수"],
)

print("\n========== 3-4. 클래스별 제품 정보 검사 ==========")
print(f"원본 제품 정보 기록: {len(original_product_metadata):,}개")
print(
    f"제품 정보를 찾은 클래스: "
    f"{original_product_metadata['drug_code'].nunique():,}종"
)
print(f"원본 연결을 찾지 못한 이미지·클래스: {len(missing_product_keys):,}건")
print(f"원본 읽기·구조 오류: {len(product_read_error_rows):,}건")
print(f"제품 정보 불일치 후보: {len(product_variations):,}건")
print(
    f"불일치 후보 클래스: "
    f"{product_variations['drug_code'].nunique():,}종"
)

if not product_variations.empty:
    display(product_variations)

if not product_missing_values.empty:
    print("\n========== 빈 제품 정보: 최대 20건 ==========")
    display(product_missing_values.head(20))

if missing_product_keys:
    print("\n========== 원본 연결 누락: 최대 20건 ==========")
    display(pd.DataFrame(
        sorted(missing_product_keys)[:20],
        columns=["source_number", "original_image", "drug_code"],
    ))

if product_read_error_rows:
    display(pd.DataFrame(product_read_error_rows).head(20))

if (
    product_variations.empty
    and not missing_product_keys
    and not product_read_error_rows
):
    print(
        "\n[체크 가능] 동일 클래스에 여러 제품명·"
        "상충하는 외형 정보 — 수집된 비어 있지 않은 값에서 없음"
    )
else:
    print("\n불일치 후보 또는 원본 연결 문제 확인이 필요합니다.")

print("\n※ 빈 각인은 실제로 각인이 없는 경우도 있어 오류로 단정하지 않습니다.")
print("※ 값이 여러 종류면 검토 후보이며 즉시 오류로 확정하지 않습니다.")
print("※ 파일 수정·삭제 없음")

### 3-5. 파일명 기준 알약 개수와 annotation 개수 비교

In [ ]:
# 3-5. 파일명 기준 알약 개수와 annotation 개수 비교
# 점검표:
# [교차 검증 / 의심] 실제 알약 개수와 Annotation 개수 불일치

pill_count_checks = filename_composition_checks[[
    "json_file",
    "image_id",
    "original_image",
    "combination",
    "filename_format_ok",
    "expected_codes",
    "annotation_count",
]].copy()

# 파일명 형식이 유효한 경우에만 기대 개수 계산
pill_count_checks["expected_count"] = [
    len(codes) if format_ok else pd.NA
    for codes, format_ok in zip(
        pill_count_checks["expected_codes"],
        pill_count_checks["filename_format_ok"],
    )
]
pill_count_checks["expected_count"] = (
    pill_count_checks["expected_count"].astype("Int64")
)

pill_count_checks["count_match"] = (
    pill_count_checks["expected_count"]
    == pill_count_checks["annotation_count"]
)

# 1. 파일명 기대 개수와 annotation 수 불일치
pill_count_issues = pill_count_checks.loc[
    pill_count_checks["count_match"].eq(False)
].copy()

pill_count_unchecked = pill_count_checks.loc[
    pill_count_checks["expected_count"].isna()
].copy()

# 2. 같은 조합의 사진끼리 annotation 수가 다른지 확인
# 코드 정렬 순서와 관계없이 같은 조합으로 묶음
pill_count_checks["combination_key"] = (
    pill_count_checks["expected_codes"]
    .map(lambda codes: tuple(sorted(codes)))
)

valid_count_checks = pill_count_checks.loc[
    pill_count_checks["filename_format_ok"]
]

combination_count_summary = (
    valid_count_checks.groupby("combination_key")
    .agg(
        image_count=("image_id", "size"),
        annotation_count_types=(
            "annotation_count",
            lambda values: sorted(set(values)),
        ),
        annotation_count_variations=("annotation_count", "nunique"),
    )
    .reset_index()
)

combination_count_issues = combination_count_summary.loc[
    combination_count_summary["annotation_count_variations"] > 1
].copy()

inconsistent_combination_keys = set(
    combination_count_issues["combination_key"]
)

combination_count_issue_images = pill_count_checks.loc[
    pill_count_checks["combination_key"].isin(
        inconsistent_combination_keys
    )
].copy()

print("========== 3-5. 알약 기대 개수·annotation 수 검사 ==========")
print(f"검사 이미지: {len(pill_count_checks):,}장")
print(f"기대 개수 확인 불가: {len(pill_count_unchecked):,}장")
print(f"기대 개수와 annotation 수 불일치: {len(pill_count_issues):,}장")
print(
    f"사진별 annotation 수가 다른 조합: "
    f"{len(combination_count_issues):,}개"
)
print(
    f"해당 조합에 포함된 이미지: "
    f"{len(combination_count_issue_images):,}장"
)

if not pill_count_issues.empty:
    print("\n========== 개수 불일치: 최대 20장 ==========")
    display(pill_count_issues[[
        "json_file",
        "original_image",
        "expected_count",
        "annotation_count",
    ]].head(20))

if not combination_count_issues.empty:
    print("\n========== 같은 조합 내 개수 차이 ==========")
    display(combination_count_issues.head(20))

    print("\n========== 해당 이미지: 최대 20장 ==========")
    display(combination_count_issue_images[[
        "json_file",
        "original_image",
        "annotation_count",
    ]].head(20))

if not pill_count_unchecked.empty:
    print("\n========== 기대 개수 확인 불가 ==========")
    display(pill_count_unchecked[[
        "json_file", "original_image"
    ]].head(20))

if (
    pill_count_issues.empty
    and combination_count_issues.empty
    and pill_count_unchecked.empty
):
    print("\n[체크 가능] 파일명 기준 알약 개수와 annotation 수 불일치 — 없음")
    print("[체크 가능] 같은 조합의 사진 간 annotation 수 차이 — 없음")

print("\n※ 사진 속 실제 알약 개수를 직접 센 결과는 아닙니다.")
print("※ 개수가 같아도 박스 위치가 올바르다는 뜻은 아닙니다.")
print("※ 제외 예정 94장을 포함한 전체 검사입니다.")
print("※ 파일 수정·삭제 없음")

### 3-6. 실물 길이 대비 bbox 크기 검사

In [ ]:
# 3-6. 실물 길이 대비 bbox 크기 검사
# 점검표: [교차 검증 / 의심] 실제 알약 위치와 bbox 불일치

PIXELS_PER_MM_MIN = 18
PIXELS_PER_MM_MAX = 33

metadata_keys = [
    "source_number", "original_image", "drug_code"
]

# 1. 원본의 실물 긴 축 길이 정리
length_metadata = original_product_metadata[
    metadata_keys + ["leng_long"]
].copy()

length_metadata["source_number"] = (
    length_metadata["source_number"].astype(int)
)
length_metadata["original_image"] = (
    length_metadata["original_image"]
    .map(lambda name: Path(name).name)
)
length_metadata["length_mm"] = pd.to_numeric(
    length_metadata["leng_long"], errors="coerce"
)

# 같은 이미지·클래스의 원본 기록이 여러 개여도
# 서로 다른 유효 길이가 있으면 임의로 하나를 선택하지 않음
length_lookup = (
    length_metadata.groupby(metadata_keys)
    .agg(
        length_values=(
            "length_mm",
            lambda values: sorted({
                float(value)
                for value in values
                if is_finite_number(value) and value > 0
            }),
        )
    )
    .reset_index()
)

length_lookup["length_mm"] = length_lookup["length_values"].map(
    lambda values: values[0] if len(values) == 1 else float("nan")
)
length_lookup["length_status"] = length_lookup["length_values"].map(
    lambda values: (
        "정상"
        if len(values) == 1
        else "유효한 길이 없음"
        if len(values) == 0
        else "원본 길이 정보 상충"
    )
)

# 2. 선정본 annotation에 원본 길이 연결
ratio_sources = sources_df[[
    "_json_file", "source_number", "original_image"
]].copy()
ratio_sources["source_number"] = (
    ratio_sources["source_number"].astype(int)
)
ratio_sources["original_image"] = (
    ratio_sources["original_image"]
    .map(lambda name: Path(name).name)
)

bbox_ratio_checks = (
    annotations_df[[
        "_json_file", "id", "image_id", "drug_code", "bbox"
    ]]
    .rename(columns={"id": "annotation_id"})
    .merge(
        ratio_sources,
        on="_json_file",
        how="left",
        validate="many_to_one",
    )
    .merge(
        length_lookup,
        on=metadata_keys,
        how="left",
        validate="many_to_one",
    )
)

bbox_ratio_checks["length_status"] = (
    bbox_ratio_checks["length_status"]
    .fillna("원본 길이 정보 연결 실패")
)

# 앞 단계에서 형식·크기가 정상임을 확인한 bbox 사용
bbox_ratio_checks["bbox_long_px"] = (
    bbox_ratio_checks["bbox"]
    .map(lambda bbox: max(float(bbox[2]), float(bbox[3])))
)

bbox_ratio_checks["pixels_per_mm"] = (
    bbox_ratio_checks["bbox_long_px"]
    / bbox_ratio_checks["length_mm"]
)

ratio_valid = bbox_ratio_checks["length_status"].eq("정상")

bbox_ratio_unchecked = bbox_ratio_checks.loc[
    ~ratio_valid
].copy()

bbox_ratio_candidates = bbox_ratio_checks.loc[
    ratio_valid
    & (
        (bbox_ratio_checks["pixels_per_mm"] < PIXELS_PER_MM_MIN)
        | (bbox_ratio_checks["pixels_per_mm"] > PIXELS_PER_MM_MAX)
    )
].copy()

print("========== 3-6. 실물 길이 대비 bbox 크기 검사 ==========")
print(f"전체 annotation: {len(bbox_ratio_checks):,}개")
print(f"비율 계산 가능: {int(ratio_valid.sum()):,}개")
print(f"계산 불가: {len(bbox_ratio_unchecked):,}개")
print(f"크기 의심 후보: {len(bbox_ratio_candidates):,}개")
print(
    f"후보가 있는 이미지: "
    f"{bbox_ratio_candidates['image_id'].nunique():,}장"
)

print("\n========== 계산된 비율 분포: 단위 px/mm ==========")
display(
    bbox_ratio_checks.loc[ratio_valid, "pixels_per_mm"]
    .describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
    .to_frame()
)

if not bbox_ratio_candidates.empty:
    print("\n========== 크기 의심 후보: 최대 20건 ==========")
    display(
        bbox_ratio_candidates.sort_values("pixels_per_mm")[[
            "_json_file",
            "annotation_id",
            "drug_code",
            "bbox",
            "length_mm",
            "pixels_per_mm",
        ]].head(20)
    )

if not bbox_ratio_unchecked.empty:
    print("\n========== 계산 불가 사유 ==========")
    display(
        bbox_ratio_unchecked["length_status"]
        .value_counts()
        .rename_axis("사유")
        .reset_index(name="Annotation 수")
    )
    display(bbox_ratio_unchecked[[
        "_json_file", "annotation_id",
        "drug_code", "length_status", "length_values",
    ]].head(20))

if bbox_ratio_candidates.empty and bbox_ratio_unchecked.empty:
    print("\n검사 완료 — 설정한 크기 비율 기준에서 의심 후보 없음")
else:
    print("\n자동 검사 완료 — 후보 또는 계산 불가 항목 확인 필요")

print("\n※ 18~33은 점검표의 참고 기준이며 자동 제외 기준이 아닙니다.")
print("※ 촬영 배율·기울기·알약 회전에 따라 비율이 달라질 수 있습니다.")
print("※ 크기가 정상이어도 박스 위치와 클래스가 맞다는 뜻은 아닙니다.")
print("※ 제외 예정 94장을 포함한 전체 검사입니다.")
print("※ 파일 수정·삭제 없음")

In [ ]:
# 3-6 추가 확인: 계산 불가 원인과 기존 제외 대상 중복

# 1. 계산 불가를 클래스별로 요약
length_unavailable_summary = (
    bbox_ratio_unchecked.groupby(
        ["drug_code", "length_status"]
    )
    .agg(
        annotation_count=("annotation_id", "size"),
        image_count=("image_id", "nunique"),
    )
    .reset_index()
)

# 원본 leng_long의 실제 저장값 확인
raw_length_values = (
    original_product_metadata.assign(
        raw_length_value=original_product_metadata["leng_long"].map(repr)
    )
    .groupby("drug_code")["raw_length_value"]
    .agg(lambda values: sorted(set(values)))
    .rename("original_length_values")
    .reset_index()
)

length_unavailable_summary = length_unavailable_summary.merge(
    raw_length_values,
    on="drug_code",
    how="left",
    validate="many_to_one",
)

print("========== 계산 불가의 클래스별 원인 ==========")
display(
    length_unavailable_summary.sort_values(
        "annotation_count", ascending=False
    )
)

# 2. 크기 의심 후보 중 기존 제외 예정 94장 구분
known_excluded_files = set(overlap_candidates["json_file"])

bbox_ratio_candidates["already_excluded"] = (
    bbox_ratio_candidates["_json_file"].isin(known_excluded_files)
)

additional_bbox_ratio_candidates = bbox_ratio_candidates.loc[
    ~bbox_ratio_candidates["already_excluded"]
].copy()

print("\n========== 크기 의심 후보 분리 ==========")
print(
    "기존 제외 대상과 겹치는 이미지: "
    f"{bbox_ratio_candidates.loc[
        bbox_ratio_candidates['already_excluded'], 'image_id'
    ].nunique():,}장"
)
print(
    "추가 검토 후보 이미지: "
    f"{additional_bbox_ratio_candidates['image_id'].nunique():,}장"
)
print(
    "추가 검토 후보 annotation: "
    f"{len(additional_bbox_ratio_candidates):,}개"
)

print("\n========== 추가 후보의 클래스별 분포 ==========")
display(
    additional_bbox_ratio_candidates.groupby("drug_code")
    .agg(
        annotation_count=("annotation_id", "size"),
        image_count=("image_id", "nunique"),
        min_ratio=("pixels_per_mm", "min"),
        max_ratio=("pixels_per_mm", "max"),
    )
    .sort_values("annotation_count", ascending=False)
    .reset_index()
)

print("\n※ 계산 불가는 bbox 오류 또는 제외 사유로 확정하지 않습니다.")
print("※ 파일 수정·삭제 없음")

- K-016548 878개, K-010487 528개 annotation은 원본 leng_long=0.0으로 실물 길이 대비 bbox 크기를 계산할 수 없어 해당 검사에서 판정 불가로 처리한다. 이 사유만으로는 제외하지 않는다.

In [ ]:
def show_bbox_size_candidates(candidates, page_size=3):
    """이미지별 크기 의심 annotation을 강조해 전체 출력."""
    review_files = (
        candidates.groupby("_json_file")["pixels_per_mm"]
        .agg(
            lambda values: max(
                abs(value - 25.3) for value in values
            )
        )
        .sort_values(ascending=False)
        .index
        .tolist()
    )

    total = len(review_files)

    if total == 0:
        print("추가 시각 검토 대상이 없습니다.")
        return

    for start in range(0, total, page_size):
        batch = review_files[start:start + page_size]
        fig, axes = plt.subplots(
            1, len(batch),
            figsize=(6 * len(batch), 8),
            squeeze=False,
        )

        for column, json_file in enumerate(batch):
            ax = axes[0, column]
            review_number = start + column + 1

            selected = candidates.loc[
                candidates["_json_file"].eq(json_file)
            ]
            candidate_ids = set(selected["annotation_id"])

            # 저장된 현재 JSON을 읽음
            with (SELECT_DIR / json_file).open(
                encoding="utf-8-sig"
            ) as f:
                document = json.load(f)

            image_path = resolve_review_image(
                document["images"][0]["file_name"]
            )

            with Image.open(image_path) as image:
                ax.imshow(image.convert("RGB"))

            # 일반 박스를 먼저, 의심 박스를 마지막에 표시
            for annotation in sorted(
                document["annotations"],
                key=lambda ann: ann["id"] in candidate_ids,
            ):
                x, y, w, h = annotation["bbox"]
                is_candidate = annotation["id"] in candidate_ids

                ax.add_patch(Rectangle(
                    (x, y), w, h,
                    fill=False,
                    edgecolor="red" if is_candidate else "gray",
                    linewidth=2.5 if is_candidate else 1,
                ))

                if is_candidate:
                    ax.text(
                        x, y,
                        f"ID {annotation['id']}\n"
                        f"{annotation['drug_code']}",
                        color="red",
                        fontsize=8,
                        va="bottom",
                        bbox=dict(
                            facecolor="white",
                            alpha=0.8,
                            edgecolor="none",
                        ),
                    )

            ax.set_title(
                f"Review {review_number}\n"
                + " / ".join(
                    f"ID {row['annotation_id']}: "
                    f"{row['pixels_per_mm']:.2f} px/mm"
                    for _, row in selected.iterrows()
                ),
                fontsize=10,
            )
            ax.axis("off")

            print(f"\n검토 {review_number}: {json_file}")
            display(selected[[
                "annotation_id",
                "drug_code",
                "length_mm",
                "pixels_per_mm",
            ]])

        plt.tight_layout()
        plt.show()
        plt.close(fig)


# 번호와 파일의 대응을 남기기 위한 목록
bbox_size_review_order = (
    additional_bbox_ratio_candidates.groupby("_json_file")[
        "pixels_per_mm"
    ]
    .agg(lambda values: max(abs(value - 25.3) for value in values))
    .sort_values(ascending=False)
    .reset_index(name="review_priority")
)
bbox_size_review_order.insert(
    0, "review_number",
    range(1, len(bbox_size_review_order) + 1),
)

show_bbox_size_candidates(
    additional_bbox_ratio_candidates,
    page_size=3,
)

In [ ]:
# 3-6 추가 검토: 박스가 비교적 정상인 14장의 클래스 정보 대조

class_review_numbers = [
    4, 6, 7, 11, 13, 14, 17,
    23, 25, 29, 33, 37, 38, 39,
]

# 시각화 때 사용한 검토번호와 파일 연결을 그대로 사용
class_review_files = bbox_size_review_order.loc[
    bbox_size_review_order["review_number"].isin(
        class_review_numbers
    ),
    ["review_number", "_json_file"],
]

class_review_targets = (
    additional_bbox_ratio_candidates.merge(
        class_review_files,
        on="_json_file",
        how="inner",
        validate="many_to_one",
    )
)

# 클래스별 검토번호와 비율 요약
class_review_summary = (
    class_review_targets.groupby("drug_code")
    .agg(
        review_numbers=(
            "review_number",
            lambda values: sorted(set(values)),
        ),
        annotation_ids=(
            "annotation_id",
            lambda values: sorted(set(values)),
        ),
        length_mm_values=(
            "length_mm",
            lambda values: sorted(set(values)),
        ),
        min_ratio=("pixels_per_mm", "min"),
        max_ratio=("pixels_per_mm", "max"),
    )
    .reset_index()
)

# 해당 14장에 연결된 원본 제품 정보만 추출
product_keys = [
    "source_number", "original_image", "drug_code"
]

target_product_keys = class_review_targets[
    product_keys
].drop_duplicates()

class_review_metadata = original_product_metadata.merge(
    target_product_keys,
    on=product_keys,
    how="inner",
    validate="many_to_one",
)

reference_fields = [
    "dl_name", "drug_shape",
    "color_class1", "color_class2",
    "print_front", "print_back",
]


def summarize_product_values(values):
    """빈 값은 별도 표시하고 서로 다른 원본 값은 모두 유지."""
    nonempty = sorted({
        str(value).strip()
        for value in values
        if pd.notna(value) and str(value).strip()
    })
    return " / ".join(nonempty) if nonempty else "(정보 없음)"


class_product_reference = (
    class_review_metadata.groupby("drug_code")[reference_fields]
    .agg(summarize_product_values)
    .reset_index()
)

class_review_reference = class_review_summary.merge(
    class_product_reference,
    on="drug_code",
    how="left",
    validate="one_to_one",
)

print("========== 3-6. 클래스·외형 대조 자료 ==========")
print(
    f"대상 이미지: "
    f"{class_review_targets['_json_file'].nunique()}장"
)
print(
    f"대상 클래스: "
    f"{class_review_targets['drug_code'].nunique()}종"
)

# 클래스별로 나눠 출력해 제품 정보가 잘리지 않도록 표시
for _, row in class_review_reference.iterrows():
    print(
        f"\n{row['drug_code']} | "
        f"검토번호 {row['review_numbers']}"
    )

    display(pd.DataFrame([
        {"항목": "제품명", "원본 정보": row["dl_name"]},
        {"항목": "모양", "원본 정보": row["drug_shape"]},
        {"항목": "색상 1", "원본 정보": row["color_class1"]},
        {"항목": "색상 2", "원본 정보": row["color_class2"]},
        {"항목": "앞면 각인", "원본 정보": row["print_front"]},
        {"항목": "뒷면 각인", "원본 정보": row["print_back"]},
        {"항목": "실물 긴 축 길이(mm)", "원본 정보": row["length_mm_values"]},
        {
            "항목": "검출 비율 범위(px/mm)",
            "원본 정보": (
                f"{row['min_ratio']:.2f} ~ "
                f"{row['max_ratio']:.2f}"
            ),
        },
    ]))

print("\n※ 원본 제품 정보를 출력했으며 자동 판정·수정은 하지 않았습니다.")

| 검토번호 | 대조 결과 | 권장 처리 |
|---|---|---|
| 4 | `K-029667`은 노란 원형·각인 `203`인데 큰 타원형 알약에 붙음. `K-030308`도 원본은 타원형인데 원형 알약에 붙음 | 제외 |
| 6·7 | `K-020238`은 분홍 원형·각인 `75 / 1171`인데 사진의 대상과 불일치. 6번의 `K-032310`도 갈색 연질캡슐 정보와 불일치 | 제외 |
| 11·33 | `K-016262`는 분홍 원형·각인 `ZD452220`인데 대상의 외형·각인이 다름 | 제외 |
| 13·14·17 | `K-030308`은 타원형·각인 `D2/850`인데 원형 알약에 붙음 | 제외 |
| 23·25·29 | `K-022074`는 원형·각인 `LPL / 20`인데 사진은 길쭉한 정제 | 제외 |
| **37·38·39** | **하얀 캡슐·각인 `DWP L250`이 원본 정보와 일치하고 박스도 적절함** | **유지** |

---
---
### 4-1. 클래스별 bbox 크기·종횡비 이상치 분석


In [ ]:
# 4-1. 클래스별 bbox 크기·종횡비 이상치
# 점검표:
# [EDA / 편향 / 분석] 클래스별 bbox 크기·종횡비 이상치

bbox_distribution_checks = annotations_df[[
    "_json_file", "id", "image_id", "drug_code", "bbox"
]].copy()

bbox_distribution_checks = bbox_distribution_checks.rename(
    columns={"id": "annotation_id"}
)

bbox_distribution_checks["bbox_area"] = (
    bbox_distribution_checks["bbox"]
    .map(lambda bbox: float(bbox[2]) * float(bbox[3]))
)
bbox_distribution_checks["aspect_ratio"] = (
    bbox_distribution_checks["bbox"]
    .map(lambda bbox: float(bbox[2]) / float(bbox[3]))
)

# 1. 클래스별 IQR 기준 계산
for metric in ["bbox_area", "aspect_ratio"]:
    grouped = bbox_distribution_checks.groupby("drug_code")[metric]

    q1 = grouped.transform(lambda values: values.quantile(0.25))
    q3 = grouped.transform(lambda values: values.quantile(0.75))
    iqr = q3 - q1

    bbox_distribution_checks[f"{metric}_lower"] = q1 - 1.5 * iqr
    bbox_distribution_checks[f"{metric}_upper"] = q3 + 1.5 * iqr

    bbox_distribution_checks[f"{metric}_outlier"] = (
        (bbox_distribution_checks[metric] < q1 - 1.5 * iqr)
        | (bbox_distribution_checks[metric] > q3 + 1.5 * iqr)
    )

bbox_distribution_checks["any_outlier"] = (
    bbox_distribution_checks["bbox_area_outlier"]
    | bbox_distribution_checks["aspect_ratio_outlier"]
)

# 2. 기존에 검토한 이미지 구분
known_overlap_files = set(overlap_candidates["json_file"])
reviewed_size_files = set(bbox_size_review_order["_json_file"])

bbox_distribution_checks["overlap_exclusion_decided"] = (
    bbox_distribution_checks["_json_file"].isin(known_overlap_files)
)
bbox_distribution_checks["reviewed_in_3_6"] = (
    bbox_distribution_checks["_json_file"].isin(reviewed_size_files)
)

bbox_distribution_candidates = bbox_distribution_checks.loc[
    bbox_distribution_checks["any_outlier"]
].copy()

# 앞서 검토한 이미지를 제외한 새로운 후보
new_bbox_distribution_candidates = (
    bbox_distribution_candidates.loc[
        ~bbox_distribution_candidates["overlap_exclusion_decided"]
        & ~bbox_distribution_candidates["reviewed_in_3_6"]
    ].copy()
)

# 3. 클래스별 요약
bbox_distribution_summary = (
    bbox_distribution_checks.groupby("drug_code")
    .agg(
        annotation_count=("annotation_id", "size"),
        median_area=("bbox_area", "median"),
        median_aspect_ratio=("aspect_ratio", "median"),
        area_outlier_count=("bbox_area_outlier", "sum"),
        aspect_outlier_count=("aspect_ratio_outlier", "sum"),
        any_outlier_count=("any_outlier", "sum"),
    )
    .reset_index()
)

bbox_distribution_summary["outlier_rate_pct"] = (
    100
    * bbox_distribution_summary["any_outlier_count"]
    / bbox_distribution_summary["annotation_count"]
)

print("========== 4-1. 클래스별 bbox 이상치 분석 ==========")
print(f"분석 annotation: {len(bbox_distribution_checks):,}개")
print(
    f"분석 클래스: "
    f"{bbox_distribution_checks['drug_code'].nunique():,}종"
)
print(
    f"면적 이상치: "
    f"{bbox_distribution_checks['bbox_area_outlier'].sum():,}개"
)
print(
    f"종횡비 이상치: "
    f"{bbox_distribution_checks['aspect_ratio_outlier'].sum():,}개"
)
print(f"둘 중 하나 이상 해당: {len(bbox_distribution_candidates):,}개")
print(
    f"해당 이미지: "
    f"{bbox_distribution_candidates['image_id'].nunique():,}장"
)

print("\n========== 기존 검토 이미지와 구분 ==========")
print(
    f"기존 중첩 제외 결정 94장에 속하는 후보 이미지: "
    f"{bbox_distribution_candidates.loc[
        bbox_distribution_candidates['overlap_exclusion_decided'],
        'image_id'
    ].nunique():,}장"
)
print(
    f"3-6에서 검토한 39장에 속하는 후보 이미지: "
    f"{bbox_distribution_candidates.loc[
        bbox_distribution_candidates['reviewed_in_3_6'],
        'image_id'
    ].nunique():,}장"
)
print(
    f"새로운 이상치 후보 이미지: "
    f"{new_bbox_distribution_candidates['image_id'].nunique():,}장"
)

print("\n========== 이상치 비율이 높은 클래스: 최대 20종 ==========")
display(
    bbox_distribution_summary.loc[
        bbox_distribution_summary["any_outlier_count"] > 0
    ]
    .sort_values(
        ["outlier_rate_pct", "any_outlier_count"],
        ascending=False,
    )
    .head(20)
)

print("\n========== 새로운 후보 표본: 최대 20개 ==========")
display(new_bbox_distribution_candidates[[
    "_json_file",
    "annotation_id",
    "drug_code",
    "bbox",
    "bbox_area",
    "bbox_area_lower",
    "bbox_area_upper",
    "aspect_ratio",
    "aspect_ratio_lower",
    "aspect_ratio_upper",
    "bbox_area_outlier",
    "aspect_ratio_outlier",
]].head(20))

print("\n[분석 완료] 클래스별 bbox 크기·종횡비 이상치")
print("※ 이상치는 통계적 특성이며 라벨 오류 확정이 아닙니다.")
print("※ 알약 회전·촬영 각도 차이도 종횡비 이상치로 잡힐 수 있습니다.")
print("※ 표본이 적거나 IQR이 0인 클래스는 해석에 주의해야 합니다.")
print("※ 제외 전 전체 데이터의 분포이며 최종 제외 후 달라질 수 있습니다.")
print("※ 파일 수정·삭제 없음")

In [ ]:
# 4-1 표본 시각 검토 — 유형별 최대 3개, 총 최대 9개

sample_types = [
    ("종횡비만 이상치", False, True),
    ("면적만 이상치", True, False),
    ("면적·종횡비 모두 이상치", True, True),
]

sample_parts = []
used_sample_files = set()

for label, area_flag, aspect_flag in sample_types:
    pool = new_bbox_distribution_candidates.loc[
        new_bbox_distribution_candidates["bbox_area_outlier"].eq(
            area_flag
        )
        & new_bbox_distribution_candidates["aspect_ratio_outlier"].eq(
            aspect_flag
        )
        & ~new_bbox_distribution_candidates["_json_file"].isin(
            used_sample_files
        )
    ].sample(frac=1, random_state=42)

    selected_indices = []
    used_classes = set()
    selected_files = set()

    # 가능한 한 서로 다른 클래스·이미지 선택
    for index, row in pool.iterrows():
        if (
            row["drug_code"] in used_classes
            or row["_json_file"] in selected_files
        ):
            continue

        selected_indices.append(index)
        used_classes.add(row["drug_code"])
        selected_files.add(row["_json_file"])

        if len(selected_indices) == 3:
            break

    # 클래스 수가 부족하면 같은 클래스의 다른 이미지로 보충
    if len(selected_indices) < 3:
        for index, row in pool.iterrows():
            if row["_json_file"] in selected_files:
                continue

            selected_indices.append(index)
            selected_files.add(row["_json_file"])

            if len(selected_indices) == 3:
                break

    selected = pool.loc[selected_indices].copy()
    selected["sample_type"] = label
    sample_parts.append(selected)
    used_sample_files.update(selected_files)

iqr_review_samples = pd.concat(
    sample_parts, ignore_index=True
)
iqr_review_samples.insert(
    0, "review_number", range(1, len(iqr_review_samples) + 1)
)

print(f"표본 검토 대상: {len(iqr_review_samples)}개 박스")
print("※ 표본 수가 부족한 유형은 가능한 수만 표시합니다.")

for start in range(0, len(iqr_review_samples), 3):
    batch = iqr_review_samples.iloc[start:start + 3]

    fig, axes = plt.subplots(
        1, len(batch),
        figsize=(6 * len(batch), 8),
        squeeze=False,
    )

    for column, (_, sample) in enumerate(batch.iterrows()):
        ax = axes[0, column]

        # 디스크에 저장된 현재 라벨 사용
        with (SELECT_DIR / sample["_json_file"]).open(
            encoding="utf-8-sig"
        ) as f:
            document = json.load(f)

        image_path = resolve_review_image(
            document["images"][0]["file_name"]
        )
        with Image.open(image_path) as image:
            ax.imshow(image.convert("RGB"))

        for annotation in sorted(
            document["annotations"],
            key=lambda ann: ann["id"] == sample["annotation_id"],
        ):
            x, y, w, h = annotation["bbox"]
            selected = annotation["id"] == sample["annotation_id"]

            ax.add_patch(Rectangle(
                (x, y), w, h,
                fill=False,
                edgecolor="red" if selected else "gray",
                linewidth=2.5 if selected else 1,
            ))

            if selected:
                ax.text(
                    x, y,
                    f"ID {annotation['id']} | {annotation['drug_code']}",
                    color="red",
                    fontsize=8,
                    va="bottom",
                    bbox=dict(
                        facecolor="white",
                        alpha=0.8,
                        edgecolor="none",
                    ),
                )

        ax.set_title(
            f"Sample {sample['review_number']}\n"
            f"{sample['drug_code']}\n"
            f"area={sample['bbox_area']:.0f}, "
            f"w/h={sample['aspect_ratio']:.3f}",
            fontsize=10,
        )
        ax.axis("off")

        print(
            f"\n표본 {sample['review_number']} | "
            f"{sample['sample_type']} | {sample['drug_code']}"
        )
        print(f"JSON: {sample['_json_file']}")

    plt.tight_layout()
    plt.show()
    plt.close(fig)

print("\n※ 표본에서 정상이어도 전체 이상치가 정상이라고 확정하지 않습니다.")
print("※ 파일 수정·삭제 없음")

### 클래스별 bbox 크기·종횡비 이상치 분석 결과

- 전체 annotation 39,392개 중 면적 또는 종횡비 이상치는
  1,590개이며, 해당 이미지는 1,489장이다.
- 기존 검토 이미지를 제외한 후보에서 유형별 3개씩,
  총 9개 박스를 표본 시각 검토했다.
- 표본 1~8번은 박스가 실제 알약을 대체로 잘 감쌌다.
- 표본 9번은 박스 위치·크기 오류를 확인하여 추가 제외 대상으로 기록한다.
- 통계적 이상치만으로 나머지 후보를 일괄 제외하지 않는다.
- 전체 후보의 정상 여부나 클래스 정확성을 확정한 결과는 아니다.

### 4-2. 클래스별 bbox 중심 위치 편향


In [ ]:
# 4-2. 클래스별 bbox 중심 위치 편향
# 점검표: [EDA / 편향 / 분석] 클래스별 bbox 중심 위치 편향

MIN_POSITION_COUNT = 10
POSITION_BIAS_THRESHOLD = 0.70

# JSON 경로와 이미지 ID를 함께 사용해 크기 연결
position_checks = annotations_df[[
    "_json_file", "id", "image_id", "drug_code", "bbox"
]].rename(columns={"id": "annotation_id"}).merge(
    images_df[[
        "_json_file", "id", "width", "height"
    ]].rename(columns={"id": "image_id"}),
    on=["_json_file", "image_id"],
    how="left",
    validate="many_to_one",
)

if position_checks[["width", "height"]].isna().any().any():
    raise ValueError("이미지 크기 연결에 실패한 annotation이 있습니다.")

# 1. 박스 중심을 이미지 너비·높이로 나눠 0~1로 정규화
position_checks["center_x"] = [
    (bbox[0] + bbox[2] / 2) / width
    for bbox, width in zip(
        position_checks["bbox"], position_checks["width"]
    )
]
position_checks["center_y"] = [
    (bbox[1] + bbox[3] / 2) / height
    for bbox, height in zip(
        position_checks["bbox"], position_checks["height"]
    )
]

# 경계값 0.5는 오른쪽 또는 아래쪽으로 분류
position_checks["quadrant"] = [
    ("상" if cy < 0.5 else "하")
    + ("좌" if cx < 0.5 else "우")
    for cx, cy in zip(
        position_checks["center_x"], position_checks["center_y"]
    )
]

quadrant_names = {
    "상좌": "좌상",
    "상우": "우상",
    "하좌": "좌하",
    "하우": "우하",
}
position_checks["quadrant"] = (
    position_checks["quadrant"].map(quadrant_names)
)

# 2. 클래스별 네 구역의 박스 수 집계
quadrant_columns = ["좌상", "우상", "좌하", "우하"]

position_summary = pd.crosstab(
    position_checks["drug_code"],
    position_checks["quadrant"],
).reindex(
    columns=quadrant_columns, fill_value=0
)

position_summary["annotation_count"] = (
    position_summary[quadrant_columns].sum(axis=1)
)
position_summary["dominant_quadrant"] = (
    position_summary[quadrant_columns].idxmax(axis=1)
)
position_summary["dominant_count"] = (
    position_summary[quadrant_columns].max(axis=1)
)
position_summary["dominant_ratio"] = (
    position_summary["dominant_count"]
    / position_summary["annotation_count"]
)

position_summary["eligible"] = (
    position_summary["annotation_count"] >= MIN_POSITION_COUNT
)
position_summary["position_biased"] = (
    position_summary["eligible"]
    & (
        position_summary["dominant_ratio"]
        >= POSITION_BIAS_THRESHOLD
    )
)

position_summary = position_summary.reset_index()

position_bias_candidates = position_summary.loc[
    position_summary["position_biased"]
].sort_values(
    "dominant_ratio", ascending=False
).copy()

position_summary["dominant_pct"] = (
    position_summary["dominant_ratio"] * 100
)
position_bias_candidates["dominant_pct"] = (
    position_bias_candidates["dominant_ratio"] * 100
)

print("========== 4-2. 클래스별 중심 위치 편향 ==========")
print(f"분석 annotation: {len(position_checks):,}개")
print(f"전체 클래스: {len(position_summary):,}종")
print(
    f"박스 {MIN_POSITION_COUNT}개 이상인 클래스: "
    f"{position_summary['eligible'].sum():,}종"
)
print(
    f"한 구역에 {POSITION_BIAS_THRESHOLD:.0%} 이상 집중된 클래스: "
    f"{len(position_bias_candidates):,}종"
)

display_columns = [
    "drug_code",
    "annotation_count",
    *quadrant_columns,
    "dominant_quadrant",
    "dominant_pct",
]

if position_bias_candidates.empty:
    print("\n설정한 기준에 해당하는 위치 편향 클래스 없음")
    print("\n========== 집중도가 높은 클래스: 최대 20종 ==========")
    display(
        position_summary.loc[position_summary["eligible"]]
        .sort_values("dominant_ratio", ascending=False)
        [display_columns]
        .head(20)
    )
else:
    print("\n========== 위치 편향 후보 ==========")
    display(position_bias_candidates[display_columns])

print("\n[분석 완료] 클래스별 bbox 중심 위치 편향")
print("※ 위치 편향은 데이터 분포 특성이며 자동 제외 사유가 아닙니다.")
print("※ 제외 전 전체 선정본 기준이며 최종 제외 후 분포는 달라질 수 있습니다.")
print("※ 이번 2×2 분석만으로 모든 형태의 위치 편향을 확인한 것은 아닙니다.")
print("※ 파일 수정·삭제 없음")

## 제외 목록과 최종 검증 기록 정리

| 제외 사유 | 이미지 수 |
|---|---:|
| 동일 bbox·과도한 중첩 | 94장 |
| 3-6에서 확인한 위치·크기·클래스 불일치 및 잘림 | 36장 |
| 4-1 표본 9번의 박스 오류 | 1장 |
| **합계** | **131장** |

In [ ]:
# 5. 재선정 — 제외 사유·유지 목록·클래스 분포·결과 기록 저장

# 1. 제외 사유 통합
exclusion_rows = []

for json_file in sorted(set(overlap_candidates["json_file"])):
    exclusion_rows.append({
        "json_file": json_file,
        "stage": "bbox 중복·중첩 검토",
        "review_number": None,
        "reason": "중복·중첩 후보 시각 검토 후 제외 결정",
    })

# 3-6: 검토 37·38·39번만 유지, 나머지 36장 제외
geometry_error_numbers = {
    1, 2, 3, 5, 8, 9, 10, 12, 15, 16, 18, 19,
    20, 21, 22, 24, 26, 27, 28, 30, 31, 34, 35, 36,
}
class_error_numbers = {
    4, 6, 7, 11, 13, 14, 17, 23, 25, 29, 33,
}

expected_review_numbers = set(range(1, 40))
if set(bbox_size_review_order["review_number"]) != expected_review_numbers:
    raise ValueError("3-6 검토번호가 1~39인지 확인해 주세요.")

for _, row in bbox_size_review_order.iterrows():
    number = int(row["review_number"])

    if number in {37, 38, 39}:
        continue

    if number in geometry_error_numbers:
        reason = "시각 검토에서 bbox 위치·크기 오류 확인"
    elif number in class_error_numbers:
        reason = "원본 제품 정보와 박스 안 알약의 외형·각인 불일치"
    elif number == 32:
        reason = "이미지 경계에서 알약이 잘린 사례로 보수적 제외"
    else:
        raise ValueError(f"처리 기준이 없는 검토번호: {number}")

    exclusion_rows.append({
        "json_file": row["_json_file"],
        "stage": "3-6",
        "review_number": number,
        "reason": reason,
    })

iqr_error_sample = iqr_review_samples.loc[
    iqr_review_samples["review_number"].eq(9)
]
if len(iqr_error_sample) != 1:
    raise ValueError("4-1 표본 9번을 하나로 확정할 수 없습니다.")

exclusion_rows.append({
    "json_file": iqr_error_sample.iloc[0]["_json_file"],
    "stage": "4-1",
    "review_number": 9,
    "reason": "표본 시각 검토에서 가느다란 영역에 놓인 잘못된 bbox 확인",
})

final_exclusion_reasons = pd.DataFrame(exclusion_rows)
final_excluded_files = set(final_exclusion_reasons["json_file"])

all_json_files = {
    path.relative_to(SELECT_DIR).as_posix()
    for path in json_paths
}

if not final_excluded_files.issubset(all_json_files):
    raise ValueError("제외 목록에 현재 선정본에 없는 JSON이 있습니다.")

# 2. 원본을 다시 읽어 현재 저장 상태 기준으로 수량 집계
selection_rows = []
class_count_rows = []

for json_file in sorted(all_json_files):
    with (SELECT_DIR / json_file).open(encoding="utf-8-sig") as f:
        document = json.load(f)

    keep = json_file not in final_excluded_files

    selection_rows.append({
        "json_file": json_file,
        "file_name": document["images"][0]["file_name"],
        "keep": keep,
        "annotation_count": len(document["annotations"]),
    })

    for annotation in document["annotations"]:
        class_count_rows.append({
            "drug_code": annotation["drug_code"],
            "keep": keep,
        })

final_selection = pd.DataFrame(selection_rows)
final_kept_images = final_selection.loc[
    final_selection["keep"]
].copy()

class_counts = pd.DataFrame(class_count_rows)
before_counts = class_counts.groupby("drug_code").size()
after_counts = (
    class_counts.loc[class_counts["keep"]]
    .groupby("drug_code")
    .size()
)

final_class_distribution = pd.DataFrame({
    "before_boxes": before_counts,
    "after_boxes": after_counts,
}).fillna(0).astype(int)

final_class_distribution["removed_boxes"] = (
    final_class_distribution["before_boxes"]
    - final_class_distribution["after_boxes"]
)

lost_classes = final_class_distribution.loc[
    final_class_distribution["after_boxes"].eq(0)
]

# 이후 학습 코드에서 사용할 실제 경로 목록
final_training_json_paths = [
    SELECT_DIR / name for name in final_kept_images["json_file"]
]

# 3. 결과 저장 — 검사 대상 select 밖에 저장
final_result_dir = (
    SELECT_DIR.parent
    / f"validation_final_{datetime.now():%Y%m%d_%H%M%S_%f}"
)
final_result_dir.mkdir(parents=True, exist_ok=False)

final_exclusion_reasons.to_csv(
    final_result_dir / "exclusion_reasons.csv",
    index=False, encoding="utf-8-sig",
)
final_selection.to_csv(
    final_result_dir / "selection_manifest.csv",
    index=False, encoding="utf-8-sig",
)
final_class_distribution.to_csv(
    final_result_dir / "class_distribution.csv",
    encoding="utf-8-sig",
)
(final_result_dir / "training_json_files.json").write_text(
    json.dumps(
        final_kept_images["json_file"].tolist(),
        ensure_ascii=False, indent=2,
    ),
    encoding="utf-8",
)

# 4-2 결과가 메모리에 있는 경우 실제 집계 반영
if "position_bias_candidates" in globals():
    position_note = (
        f"2×2 구역 중 한 구역에 70% 이상 집중된 "
        f"클래스는 {len(position_bias_candidates)}종이다. "
        "위치 편향만으로 추가 제외하지 않았다."
    )
    position_summary.to_csv(
        final_result_dir / "position_summary.csv",
        index=False, encoding="utf-8-sig",
    )
else:
    position_note = (
        "4-2 실행 결과가 현재 메모리에 없어 완료 판정을 보류한다."
    )

report = f"""# SELECT 데이터셋 검증 및 재선정 결과

## 최종 선정
- 검사 이미지: {len(final_selection):,}장
- 제외 이미지: {len(final_excluded_files):,}장
- 유지 이미지: {len(final_kept_images):,}장
- 유지 annotation: {final_kept_images['annotation_count'].sum():,}개
- 유지 클래스: {(final_class_distribution['after_boxes'] > 0).sum():,}종
- 제외 후 박스가 0개가 된 클래스: {len(lost_classes)}종

## 제외 근거
- bbox 중복·중첩 시각 검토 결과 94장 제외.
- 3-6 검토에서 위치·크기 오류 24장, 제품 정보 불일치 11장,
  이미지 경계 잘림 1장 제외.
- 4-1 표본 9번에서 bbox 오류를 확인하여 추가 제외.
- 중복된 제외 사유가 있더라도 이미지 수는 한 번만 집계했다.
- 첫 시험 보정 이미지도 최종 제외 대상에 포함했다.

## 유지 및 판정 불가
- 완전 중복 이미지는 없었다.
- pHash 유사 후보는 9쌍 표본에서 알약 구성이 다른 것을 확인했으며,
  유사도만으로 추가 제외하지 않았다.
- K-016548 878개, K-010487 528개 annotation은 원본 실물 길이가
  0으로 기록되어 실물 길이 대비 크기 검사에서 판정 불가로 처리했다.
  이 사유만으로 제외하지 않았다.
- 3-6 검토 37·38·39번은 외형·각인이 원본 정보와 일치하고
  박스가 적절하여 유지했다.
- IQR 이상치는 1,590개 annotation에서 확인했다.
  9개 표본 중 1~8번은 박스가 대체로 적절했고 9번은 오류였다.
  통계적 이상치 전체를 자동 제외하지 않았다.

## 위치 편향
- {position_note}

## 검증 범위와 사용 방법
- 전체 이미지의 누락·읽기 실패·JSON 크기 불일치는 없었다.
- 선정본과 클래스 매핑표의 ID 대조 및 파일명 구성·개수 검사에서
  불일치가 없었다.
- 매핑표의 미사용 클래스 K-003351, K-003483의 aihub_dl_idx 결측은
  임의로 채우지 않았다.
- 시각 검토는 자동 검사 후보와 표본에 한정되며,
  유지 이미지 전체의 클래스·bbox 정확성을 보장하는 결과는 아니다.
- 원본 이미지와 JSON은 삭제하지 않았다.
- 학습 데이터 구성에는 training_json_files.json의 유지 목록을 적용한다.
- 이 검증에는 학습·검증·테스트 사이의 데이터 누수 검사가 포함되지 않는다.
"""

(final_result_dir / "validation_report.md").write_text(
    report, encoding="utf-8"
)

print("========== 최종 재선정 결과 ==========")
print(f"전체 이미지: {len(final_selection):,}장")
print(f"제외 이미지: {len(final_excluded_files):,}장")
print(f"유지 이미지: {len(final_kept_images):,}장")
print(
    f"유지 annotation: "
    f"{final_kept_images['annotation_count'].sum():,}개"
)
print(
    f"유지 클래스: "
    f"{(final_class_distribution['after_boxes'] > 0).sum():,}종"
)
print(f"박스가 0개가 된 클래스: {len(lost_classes)}종")

if not lost_classes.empty:
    display(lost_classes)

print("\n클래스별 유지 박스 수가 적은 순서:")
display(final_class_distribution.sort_values("after_boxes").head(20))

print(f"\n결과 저장 위치: {final_result_dir}")
print("\n※ 학습에는 final_training_json_paths 또는 저장한 유지 목록을 사용하세요.")
print("※ 기존 select 폴더 전체를 그대로 읽으면 제외 대상도 포함됩니다.")

In [ ]:
# 제외 후 전체 클래스 분포
final_class_table = (
    final_class_distribution
    .sort_values("after_boxes", ascending=False)
    .reset_index()
    .rename(columns={
        "drug_code": "클래스",
        "before_boxes": "제외 전 박스 수",
        "after_boxes": "제외 후 박스 수",
        "removed_boxes": "제외 박스 수",
    })
)

final_class_table["제외 후 비중(%)"] = (
    final_class_table["제외 후 박스 수"]
    / final_class_table["제외 후 박스 수"].sum()
    * 100
).round(2)

print(f"전체 클래스: {len(final_class_table)}종")
print(
    f"전체 박스: "
    f"{final_class_table['제외 후 박스 수'].sum():,}개"
)

# 116종 모두 표시
with pd.option_context("display.max_rows", None):
    display(final_class_table)

# 전체 분포를 CSV로도 저장
distribution_path = (
    final_result_dir / "class_distribution_after.csv"
)
final_class_table.to_csv(
    distribution_path,
    index=False,
    encoding="utf-8-sig",
)

print(f"분포 저장 위치: {distribution_path}")

In [ ]:
plot_counts = (
    final_class_distribution["after_boxes"]
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(
    figsize=(12, max(8, len(plot_counts) * 0.25))
)

bars = ax.barh(
    plot_counts.index,
    plot_counts.values,
    color="#4472C4",
)

# 가장 많은 클래스를 맨 위에 표시
ax.invert_yaxis()
ax.set_xlim(0, plot_counts.max() * 1.15)

ax.bar_label(
    bars,
    labels=[f"{value:,}" for value in plot_counts.values],
    padding=4,
    fontsize=8,
)

ax.set_title(
    f"Class distribution after exclusion\n"
    f"{len(plot_counts)} classes | "
    f"{plot_counts.sum():,} bounding boxes",
    fontsize=14,
)
ax.set_xlabel("Bounding boxes")
ax.tick_params(axis="y", labelsize=8)
ax.grid(axis="x", alpha=0.25)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

chart_path = final_result_dir / "class_distribution_after.png"
fig.savefig(chart_path, dpi=180, bbox_inches="tight")

plt.show()
plt.close(fig)

print(f"그래프 저장 위치: {chart_path}")